# **Analyzing the Impact of Trump’s Social Media Sentiment on Crude Oil Price Movements**

This notebook fixes the critical issues from the previous version:

1. Fixes market/geopolitical keyword detection.
2. Removes noisy `pic.twitter.com` and URL artifacts before sentiment scoring.
3. Uses a stronger sentiment approach: **VADER when available**, with TextBlob fallback.
4. Separates **EDA**, **regression**, and **classification**.
5. Uses a **chronological train/test split**, not random splitting.
6. Blocks current-day oil leakage from the feature matrix.
7. Compares tweet/sentiment models against honest oil-only and dummy baselines.
8. Frames results realistically: this tests association and weak predictive signal, not causality.

> Project framing: This notebook tests whether Trump tweet signals are associated with short-term oil price movement. It does **not** claim that tweets alone control oil prices.

## 1) Setup
Run this cell first. If you are using Google Colab, upload the CSV files before running the notebook.

In [1]:
import os
import re
import sys
import math
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 240)

RANDOM_STATE = 42
TEST_SIZE = 0.20

## 2) Optional Plotly Dashboard Style
EternaCloud-inspired colors are applied to all Plotly dashboards and notebook tables. Run this before any visualization cells.


In [2]:
try:
    import plotly.express as px
    import plotly.graph_objects as go
    import plotly.io as pio
    PLOTLY_OK = True
except Exception:
    PLOTLY_OK = False
    px = None
    go = None
    pio = None

# =========================
# Purple + Gold + Blue + Pink Dashboard Theme
# This theme updates all Plotly dashboards and notebook tables.
# =========================
DASH_THEME = {
    "page_bg": "#060816",
    "panel_bg": "#0C0A1F",
    "plot_bg": "#12102A",
    "card_bg": "#171233",
    "card_bg_alt": "#21194A",
    "nav_white": "#F9F7FF",
    "text": "#F4F0FF",
    "muted_text": "#CDC4F1",
    "grid": "#3A3266",
    "border": "#54468C",
    "purple": "#8B5CF6",
    "deep_purple": "#5B21B6",
    "gold": "#F2C94C",
    "gold_deep": "#D4A017",
    "blue": "#3B82F6",
    "deep_blue": "#1D4ED8",
    "pink": "#EC4899",
    "soft_pink": "#F472B6",
    "cyan": "#60A5FA",
    "green": "#34D399",
    "red": "#FB7185",
    "neutral": "#B6ADD8",
}

DASH_COLORWAY = [
    DASH_THEME["purple"],
    DASH_THEME["gold"],
    DASH_THEME["blue"],
    DASH_THEME["pink"],
    DASH_THEME["soft_pink"],
    DASH_THEME["deep_purple"],
    DASH_THEME["cyan"],
    DASH_THEME["gold_deep"],
]

DASH_CONTINUOUS = [
    [0.00, DASH_THEME["deep_purple"]],
    [0.25, DASH_THEME["purple"]],
    [0.50, DASH_THEME["blue"]],
    [0.75, DASH_THEME["pink"]],
    [1.00, DASH_THEME["gold"]],
]

if PLOTLY_OK:
    custom_template = go.layout.Template(pio.templates['plotly_dark'])
    custom_template.layout.update(
        font=dict(family='Inter, Arial, sans-serif', size=14, color=DASH_THEME['text']),
        title=dict(font=dict(size=24, color=DASH_THEME['nav_white']), x=0.5, xanchor='center'),
        paper_bgcolor=DASH_THEME['page_bg'],
        plot_bgcolor=DASH_THEME['plot_bg'],
        colorway=DASH_COLORWAY,
        hoverlabel=dict(
            bgcolor=DASH_THEME['card_bg'],
            bordercolor=DASH_THEME['purple'],
            font=dict(color=DASH_THEME['text'], size=13)
        ),
        xaxis=dict(
            gridcolor=DASH_THEME['grid'],
            zerolinecolor=DASH_THEME['border'],
            linecolor=DASH_THEME['border'],
            tickfont=dict(color=DASH_THEME['text']),
            title=dict(font=dict(color=DASH_THEME['text']))
        ),
        yaxis=dict(
            gridcolor=DASH_THEME['grid'],
            zerolinecolor=DASH_THEME['border'],
            linecolor=DASH_THEME['border'],
            tickfont=dict(color=DASH_THEME['text']),
            title=dict(font=dict(color=DASH_THEME['text']))
        ),
        legend=dict(
            bgcolor='rgba(12,10,31,0.78)',
            bordercolor=DASH_THEME['border'],
            borderwidth=1,
            font=dict(color=DASH_THEME['text'])
        ),
        coloraxis=dict(
            colorscale=DASH_CONTINUOUS,
            colorbar=dict(tickfont=dict(color=DASH_THEME['text']))
        )
    )

    pio.templates['purple_gold_blue_pink_dark'] = custom_template
    pio.templates.default = 'purple_gold_blue_pink_dark'
    px.defaults.template = 'purple_gold_blue_pink_dark'
    px.defaults.color_discrete_sequence = DASH_COLORWAY
    px.defaults.color_continuous_scale = DASH_CONTINUOUS

def style_fig(fig, title=None, height=520):
    """Apply the purple / gold / blue / pink identity to every Plotly figure."""
    if not PLOTLY_OK:
        return fig

    fig.update_layout(
        template='purple_gold_blue_pink_dark',
        title=title or fig.layout.title.text,
        title_x=0.5,
        height=height,
        margin=dict(l=48, r=48, t=82, b=52),
        hovermode='x unified',
        paper_bgcolor=DASH_THEME['page_bg'],
        plot_bgcolor=DASH_THEME['plot_bg'],
        font=dict(color=DASH_THEME['text']),
        coloraxis=dict(colorscale=DASH_CONTINUOUS),
        shapes=[
            dict(
                type='rect', xref='paper', yref='paper', x0=-0.04, y0=-0.08, x1=1.04, y1=1.08,
                line=dict(color='rgba(139,92,246,0.18)', width=1),
                fillcolor='rgba(139,92,246,0.045)',
                layer='below'
            )
        ]
    )

    fig.update_xaxes(
        gridcolor=DASH_THEME['grid'],
        zerolinecolor=DASH_THEME['border'],
        linecolor=DASH_THEME['border']
    )
    fig.update_yaxes(
        gridcolor=DASH_THEME['grid'],
        zerolinecolor=DASH_THEME['border'],
        linecolor=DASH_THEME['border']
    )

    fig.update_traces(
        marker_line_color='rgba(249,247,255,0.20)',
        marker_line_width=0.6,
        selector=dict(type='bar')
    )

    return fig

# =========================
# Notebook Table Theme
# =========================
from IPython.display import display, HTML

TABLE_THEME = DASH_THEME

display(HTML(f"""
<style>
    body, .notebook_app, .jp-Notebook, .jp-Cell {{
        background-color: {TABLE_THEME['page_bg']} !important;
    }}

    table.dataframe {{
        background: linear-gradient(135deg, {TABLE_THEME['card_bg']} 0%, {TABLE_THEME['panel_bg']} 52%, #25174A 100%) !important;
        color: {TABLE_THEME['text']} !important;
        border-collapse: collapse !important;
        border: 1px solid {TABLE_THEME['border']} !important;
        border-radius: 16px !important;
        overflow: hidden !important;
        font-family: Inter, Arial, sans-serif !important;
        font-size: 13px !important;
        box-shadow: 0 18px 45px rgba(0,0,0,0.35), 0 0 42px rgba(139,92,246,0.14) !important;
    }}

    table.dataframe thead th {{
        background: linear-gradient(90deg, {TABLE_THEME['deep_purple']} 0%, {TABLE_THEME['purple']} 28%, {TABLE_THEME['blue']} 58%, {TABLE_THEME['pink']} 82%, {TABLE_THEME['gold']} 100%) !important;
        color: {TABLE_THEME['nav_white']} !important;
        border: 1px solid rgba(249,247,255,0.14) !important;
        font-weight: 800 !important;
        text-align: left !important;
        padding: 11px 13px !important;
        letter-spacing: 0.2px !important;
    }}

    table.dataframe tbody td, table.dataframe tbody th {{
        background-color: {TABLE_THEME['card_bg']} !important;
        color: {TABLE_THEME['text']} !important;
        border: 1px solid {TABLE_THEME['border']} !important;
        padding: 9px 12px !important;
        vertical-align: top !important;
    }}

    table.dataframe tbody tr:nth-child(even) td,
    table.dataframe tbody tr:nth-child(even) th {{
        background-color: {TABLE_THEME['card_bg_alt']} !important;
    }}

    table.dataframe tbody tr:hover td,
    table.dataframe tbody tr:hover th {{
        background-color: #2A1E57 !important;
        box-shadow: inset 3px 0 0 {TABLE_THEME['gold']} !important;
    }}

    .dataframe caption {{
        caption-side: top !important;
        color: {TABLE_THEME['gold']} !important;
        font-weight: 800 !important;
        text-align: left !important;
        padding: 8px 0 10px 0 !important;
        font-size: 15px !important;
    }}
</style>
"""))

def _theme_value_style(value):
    """Color only meaningful signal values, not every number in the notebook."""
    try:
        if pd.isna(value):
            return f"color: {TABLE_THEME['neutral']};"
    except Exception:
        pass

    if isinstance(value, str):
        clean_value = value.strip().lower()
        if clean_value in ["positive", "up", "actual up", "pred up"]:
            return f"color: {TABLE_THEME['gold']}; font-weight: 800;"
        if clean_value in ["negative", "down", "down/flat", "actual down/flat", "pred down/flat"]:
            return f"color: {TABLE_THEME['pink']}; font-weight: 800;"
        if clean_value in ["neutral", "unknown"]:
            return f"color: {TABLE_THEME['neutral']}; font-weight: 700;"
        return ""

    try:
        numeric_value = float(value)
    except Exception:
        return ""

    if numeric_value > 0:
        return f"color: {TABLE_THEME['gold']}; font-weight: 700;"
    if numeric_value < 0:
        return f"color: {TABLE_THEME['pink']}; font-weight: 700;"
    return f"color: {TABLE_THEME['neutral']};"

def _columns_to_highlight(data):
    signal_terms = [
        "return", "sentiment", "polarity", "correlation", "r2", "accuracy",
        "precision", "recall", "f1", "auc", "correct", "direction", "delta",
        "shock", "importance", "score"
    ]
    return [col for col in data.columns if any(term in str(col).lower() for term in signal_terms)]

def style_dark_table(data, caption=None, precision=4):
    """Apply the purple / gold / blue / pink table theme to a pandas DataFrame."""
    if not isinstance(data, pd.DataFrame):
        return data

    display_data = data.copy()

    styler = (
        display_data.style
        .set_table_styles([
            {"selector": "caption", "props": [
                ("caption-side", "top"),
                ("color", TABLE_THEME["gold"]),
                ("font-weight", "800"),
                ("text-align", "left"),
                ("padding", "8px 0 10px 0"),
                ("font-size", "15px"),
            ]},
            {"selector": "thead th", "props": [
                ("background", f"linear-gradient(90deg, {TABLE_THEME['deep_purple']} 0%, {TABLE_THEME['purple']} 28%, {TABLE_THEME['blue']} 58%, {TABLE_THEME['pink']} 82%, {TABLE_THEME['gold']} 100%)"),
                ("color", TABLE_THEME["nav_white"]),
                ("border", "1px solid rgba(249,247,255,0.14)"),
                ("font-weight", "800"),
                ("text-align", "left"),
                ("padding", "11px 13px"),
            ]},
            {"selector": "tbody td", "props": [
                ("background-color", TABLE_THEME["card_bg"]),
                ("color", TABLE_THEME["text"]),
                ("border", f"1px solid {TABLE_THEME['border']}"),
                ("padding", "9px 12px"),
                ("vertical-align", "top"),
            ]},
            {"selector": "tbody th", "props": [
                ("background-color", TABLE_THEME["card_bg"]),
                ("color", TABLE_THEME["muted_text"]),
                ("border", f"1px solid {TABLE_THEME['border']}"),
                ("padding", "9px 12px"),
                ("font-weight", "700"),
            ]},
            {"selector": "tbody tr:nth-child(even) td", "props": [
                ("background-color", TABLE_THEME["card_bg_alt"]),
            ]},
            {"selector": "tbody tr:hover td", "props": [
                ("background-color", "#2A1E57"),
            ]},
        ])
        .set_properties(**{
            "font-family": "Inter, Arial, sans-serif",
            "font-size": "13px",
            "white-space": "normal",
            "line-height": "1.35",
        })
        .format(precision=precision, na_rep="—")
    )

    highlight_cols = _columns_to_highlight(display_data)
    if highlight_cols:
        try:
            styler = styler.map(_theme_value_style, subset=highlight_cols)
        except AttributeError:
            styler = styler.applymap(_theme_value_style, subset=highlight_cols)

    if caption:
        styler = styler.set_caption(caption)

    return styler

def display_dark_table(data, caption=None, precision=4):
    """Use this instead of display(df) for a consistent professional table style."""
    if isinstance(data, pd.DataFrame):
        display(style_dark_table(data, caption=caption, precision=precision))
    else:
        display(data)


## 3) File Paths
The notebook looks for the expected files in Colab, local folders, and `/mnt/data`.

Expected files:
- `tweets2017-2026.csv`
- `Crude_Oil_daily.csv`
- Optional: `oil_hourly_2017.csv`

In [3]:
TWEETS_PATH_CANDIDATES = [
    '/content/tweets2017-2026.csv',
    '/content/tweets2017-2026(2).csv',
    'tweets2017-2026.csv',
    'tweets2017-2026(2).csv',
    '/mnt/data/tweets2017-2026.csv',
]

OIL_DAILY_PATH_CANDIDATES = [
    '/content/Crude_Oil_daily.csv',
    '/content/Crude_Oil_daily(2).csv',
    'Crude_Oil_daily.csv',
    'Crude_Oil_daily(2).csv',
    '/mnt/data/Crude_Oil_daily.csv',
]

OIL_HOURLY_PATH_CANDIDATES = [
    '/content/oil_hourly_2017.csv',
    '/content/oil_hourly_2017(2).csv',
    'oil_hourly_2017.csv',
    'oil_hourly_2017(2).csv',
    '/mnt/data/oil_hourly_2017.csv',
]

def first_existing(paths, required=True):
    for p in paths:
        if os.path.exists(p):
            return p
    if required:
        raise FileNotFoundError(f'None of these files were found: {paths}')
    return None

TWEETS_PATH = first_existing(TWEETS_PATH_CANDIDATES, required=True)
OIL_DAILY_PATH = first_existing(OIL_DAILY_PATH_CANDIDATES, required=True)
OIL_HOURLY_PATH = first_existing(OIL_HOURLY_PATH_CANDIDATES, required=False)

print('Tweets file:', TWEETS_PATH)
print('Daily oil file:', OIL_DAILY_PATH)
print('Hourly oil file:', OIL_HOURLY_PATH)

Tweets file: /content/tweets2017-2026.csv
Daily oil file: /content/Crude_Oil_daily.csv
Hourly oil file: /content/oil_hourly_2017.csv


## 4) Load Raw Data
This cell only loads the files. Cleaning is done in later sections.

In [4]:
df_raw = pd.read_csv(TWEETS_PATH)
oil_daily_raw = pd.read_csv(OIL_DAILY_PATH)

oil_hourly_raw = None
if OIL_HOURLY_PATH is not None:
    oil_hourly_raw = pd.read_csv(OIL_HOURLY_PATH)

print('Tweets shape:', df_raw.shape)
print('Daily oil shape:', oil_daily_raw.shape)
if oil_hourly_raw is not None:
    print('Hourly oil shape:', oil_hourly_raw.shape)

display_dark_table(df_raw.head())
display_dark_table(oil_daily_raw.head())
if oil_hourly_raw is not None:
    display_dark_table(oil_hourly_raw.head())

Tweets shape: (32094, 18)
Daily oil shape: (2252, 8)
Hourly oil shape: (89395, 7)


,id,date,platform,handle,text,favorite_count,repost_count,quote_flag,repost_flag,deleted_flag,word_count,hashtags,urls,user_mentions,media_count,media_urls,post_url,in_reply_to
0,822000000000000000.0000,2017-01-20 00:40:51+00:00,Twitter,realDonaldTrump,"Thank you for joining us at the Lincoln Memorial tonight- a very special evening! Together, we are going to MAKE AMERICA GREAT AGAIN! pic.twitter.com/5d774OCx5o",146346.0000,28852.0000,False,False,False,24.0000,—,https://twitter.com/i/web/status/822242449053614081,—,0.0000,—,https://x.com/realdonaldtrump/status/822242449053614081,—
1,822000000000000000.0000,2017-01-20 04:24:33+00:00,Twitter,realDonaldTrump,"Thank you for a wonderful evening in Washington, D.C. #Inauguration pic.twitter.com/a6xpFQTHj5",98164.0000,17221.0000,False,False,False,11.0000,#inauguration,—,—,1.0000,https://pbs.twimg.com/media/C2lkWIQUUAAQJVH.jpg,https://x.com/realdonaldtrump/status/822298747421986828,—
2,822000000000000000.0000,2017-01-20 12:31:53+00:00,Twitter,realDonaldTrump,It all begins today! I will see you at 11:00 A.M. for the swearing-in. THE MOVEMENT CONTINUES - THE WORK BEGINS!,234807.0000,58312.0000,False,False,False,21.0000,—,—,—,0.0000,—,https://x.com/realdonaldtrump/status/822421390125043713,—
3,823000000000000000.0000,2017-01-20 17:51:25+00:00,Twitter,realDonaldTrump,"Today we are not merely transferring power from one Administration to another, or from one party to another – but we are transferring...",95763.0000,16646.0000,False,False,False,23.0000,—,—,—,0.0000,—,https://x.com/realdonaldtrump/status/822501803615014918,—
4,823000000000000000.0000,2017-01-20 17:51:58+00:00,Twitter,realDonaldTrump,"power from Washington, D.C. and giving it back to you, the American People. #InaugurationDay",79061.0000,15074.0000,False,False,False,14.0000,#inaugurationday,—,—,0.0000,—,https://x.com/realdonaldtrump/status/822501939267141634,realDonaldTrump


,Date,Open,High,Low,Close,Volume,ticker,name
0,2017-01-20 00:00:00-05:00,51.4500,52.9000,51.3900,52.4200,567231,CL=F,Crude Oil Futures (CL=F)
1,2017-01-23 00:00:00-05:00,53.3300,53.4700,52.2100,52.7500,455333,CL=F,Crude Oil Futures (CL=F)
2,2017-01-24 00:00:00-05:00,52.8600,53.5600,52.6700,53.1800,520285,CL=F,Crude Oil Futures (CL=F)
3,2017-01-25 00:00:00-05:00,52.9500,53.4700,52.5600,52.7500,589709,CL=F,Crude Oil Futures (CL=F)
4,2017-01-26 00:00:00-05:00,52.9600,54.0600,52.7900,53.7800,578065,CL=F,Crude Oil Futures (CL=F)


,datetime,open,high,low,close,volume,oil_type
0,1/20/2017 0:00,54.4200,54.4300,54.3300,54.3800,0,Brent
1,1/20/2017 0:00,52.3000,52.3100,52.2400,52.2900,0,WTI
2,1/20/2017 1:00,54.3700,54.3700,54.1900,54.2100,0,Brent
3,1/20/2017 1:00,52.2800,52.2800,52.1400,52.1600,0,WTI
4,1/20/2017 2:00,54.2000,54.4500,54.1600,54.4400,0,Brent


## 5) Basic Column Validation
This prevents silent failure when expected columns are missing.

In [5]:
required_tweet_cols = ['date', 'text']
missing_tweet_cols = [c for c in required_tweet_cols if c not in df_raw.columns]
if missing_tweet_cols:
    raise ValueError(f'Missing required tweet columns: {missing_tweet_cols}')

print('Tweet columns available:')
print(df_raw.columns.tolist())
print('\nOil daily columns available:')
print(oil_daily_raw.columns.tolist())

Tweet columns available:
['id', 'date', 'platform', 'handle', 'text', 'favorite_count', 'repost_count', 'quote_flag', 'repost_flag', 'deleted_flag', 'word_count', 'hashtags', 'urls', 'user_mentions', 'media_count', 'media_urls', 'post_url', 'in_reply_to']

Oil daily columns available:
['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'ticker', 'name']


## 6) Clean Tweet Data
Main fixes:
- Do not drop rows blindly.
- Keep rows with valid `text` and `date`.
- Convert engagement columns safely.
- Standardize date columns in UTC.

In [6]:
df = df_raw.copy()

df = df.dropna(subset=['text']).copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True)
df = df.dropna(subset=['date']).copy()

optional_text_cols = ['hashtags', 'urls', 'user_mentions', 'in_reply_to', 'platform', 'handle']
for col in optional_text_cols:
    if col not in df.columns:
        df[col] = ''
    df[col] = df[col].fillna('').astype(str)

for col in ['favorite_count', 'repost_count', 'word_count', 'media_count']:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'id' not in df.columns:
    df['id'] = np.arange(len(df))

# Time features
for col in ['year', 'month', 'day', 'dayofweek', 'dayofyear', 'quarter']:
    df[col] = getattr(df['date'].dt, col)

df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
df['date_day'] = df['date'].dt.floor('D')
df['date_hour'] = df['date'].dt.floor('H')

df['tweet_length_ch'] = df['text'].astype(str).str.len()
df['tweet_length_class'] = pd.cut(
    df['tweet_length_ch'],
    bins=[-1, 130, 280, np.inf],
    labels=['short', 'medium', 'long']
)

df['engagement'] = df['favorite_count'] + df['repost_count']

def count_hashtags(row):
    h_col = str(row.get('hashtags', ''))
    h_text = str(row.get('text', ''))
    return max(h_col.count('#'), h_text.count('#'))

def count_mentions(row):
    m_col = str(row.get('user_mentions', ''))
    m_text = str(row.get('text', ''))
    return max(m_col.count('@'), m_text.count('@'))

df['hashtag_count'] = df.apply(count_hashtags, axis=1)
df['mention_count'] = df.apply(count_mentions, axis=1)
df['media_count_clean'] = df['media_count'].fillna(0).astype(int)

print('Clean tweets shape:', df.shape)
display_dark_table(df[['date', 'text', 'platform', 'favorite_count', 'repost_count', 'engagement']].head())

Clean tweets shape: (32094, 33)


,date,text,platform,favorite_count,repost_count,engagement
0,2017-01-20 00:40:51+00:00,"Thank you for joining us at the Lincoln Memorial tonight- a very special evening! Together, we are going to MAKE AMERICA GREAT AGAIN! pic.twitter.com/5d774OCx5o",Twitter,146346.0000,28852.0000,175198.0000
1,2017-01-20 04:24:33+00:00,"Thank you for a wonderful evening in Washington, D.C. #Inauguration pic.twitter.com/a6xpFQTHj5",Twitter,98164.0000,17221.0000,115385.0000
2,2017-01-20 12:31:53+00:00,It all begins today! I will see you at 11:00 A.M. for the swearing-in. THE MOVEMENT CONTINUES - THE WORK BEGINS!,Twitter,234807.0000,58312.0000,293119.0000
3,2017-01-20 17:51:25+00:00,"Today we are not merely transferring power from one Administration to another, or from one party to another – but we are transferring...",Twitter,95763.0000,16646.0000,112409.0000
4,2017-01-20 17:51:58+00:00,"power from Washington, D.C. and giving it back to you, the American People. #InaugurationDay",Twitter,79061.0000,15074.0000,94135.0000


## 7) Stronger Text Cleaning for NLP
Critical fix: remove noisy URL artifacts such as `pic.twitter.com/...`, even when the link appears without `https://`.

In [7]:
def clean_text_for_sentiment(text):
    x = str(text)
    x = re.sub(r'https?://\S+', ' ', x, flags=re.IGNORECASE)
    x = re.sub(r'www\.\S+', ' ', x, flags=re.IGNORECASE)
    x = re.sub(r'pic\.twitter\.com/\S+', ' ', x, flags=re.IGNORECASE)
    x = re.sub(r't\.co/\S+', ' ', x, flags=re.IGNORECASE)
    x = re.sub(r'@\w+', ' ', x)
    x = re.sub(r'#', '', x)
    x = re.sub(r'[^a-zA-Z\s]', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip().lower()
    return x

df['clean_text'] = df['text'].apply(clean_text_for_sentiment)

display_dark_table(df[['text', 'clean_text']].head())

,text,clean_text
0,"Thank you for joining us at the Lincoln Memorial tonight- a very special evening! Together, we are going to MAKE AMERICA GREAT AGAIN! pic.twitter.com/5d774OCx5o",thank you for joining us at the lincoln memorial tonight a very special evening together we are going to make america great again
1,"Thank you for a wonderful evening in Washington, D.C. #Inauguration pic.twitter.com/a6xpFQTHj5",thank you for a wonderful evening in washington d c inauguration
2,It all begins today! I will see you at 11:00 A.M. for the swearing-in. THE MOVEMENT CONTINUES - THE WORK BEGINS!,it all begins today i will see you at a m for the swearing in the movement continues the work begins
3,"Today we are not merely transferring power from one Administration to another, or from one party to another – but we are transferring...",today we are not merely transferring power from one administration to another or from one party to another but we are transferring
4,"power from Washington, D.C. and giving it back to you, the American People. #InaugurationDay",power from washington d c and giving it back to you the american people inaugurationday


## 8) Sentiment Scoring — VADER First, TextBlob Fallback
VADER is usually better for short social media text. If VADER is not available, the notebook falls back to TextBlob.

In [8]:
SENTIMENT_ENGINE = None

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    vader_analyzer = SentimentIntensityAnalyzer()
    SENTIMENT_ENGINE = 'VADER'
except Exception:
    try:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'vaderSentiment', '-q'])
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        vader_analyzer = SentimentIntensityAnalyzer()
        SENTIMENT_ENGINE = 'VADER'
    except Exception:
        vader_analyzer = None
        SENTIMENT_ENGINE = 'TextBlob'

if SENTIMENT_ENGINE == 'TextBlob':
    try:
        from textblob import TextBlob
    except Exception:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'textblob', '-q'])
        from textblob import TextBlob

print('Sentiment engine:', SENTIMENT_ENGINE)

Sentiment engine: VADER


In [9]:
def sentiment_scores(text):
    text = str(text)
    if SENTIMENT_ENGINE == 'VADER':
        scores = vader_analyzer.polarity_scores(text)
        return pd.Series({
            'polarity': scores['compound'],
            'positive_score': scores['pos'],
            'neutral_score': scores['neu'],
            'negative_score': scores['neg'],
            'subjectivity': np.nan
        })
    else:
        blob = TextBlob(text)
        return pd.Series({
            'polarity': blob.sentiment.polarity,
            'positive_score': max(blob.sentiment.polarity, 0),
            'neutral_score': 1 - abs(blob.sentiment.polarity),
            'negative_score': abs(min(blob.sentiment.polarity, 0)),
            'subjectivity': blob.sentiment.subjectivity
        })

def sentiment_label(score):
    if score > 0.05:
        return 'Positive'
    elif score < -0.05:
        return 'Negative'
    return 'Neutral'

sentiment_df = df['clean_text'].apply(sentiment_scores)
df = pd.concat([df, sentiment_df], axis=1)
df['subjectivity'] = df['subjectivity'].fillna(0)
df['sentiment'] = df['polarity'].apply(sentiment_label)
df['sentiment_score'] = df['sentiment'].map({'Positive': 1, 'Neutral': 0, 'Negative': -1})

print(df['sentiment'].value_counts())
display_dark_table(df[['text', 'clean_text', 'polarity', 'sentiment']].head())

sentiment
Positive    15122
Negative     9253
Neutral      7719
Name: count, dtype: int64


,text,clean_text,polarity,sentiment
0,"Thank you for joining us at the Lincoln Memorial tonight- a very special evening! Together, we are going to MAKE AMERICA GREAT AGAIN! pic.twitter.com/5d774OCx5o",thank you for joining us at the lincoln memorial tonight a very special evening together we are going to make america great again,0.8622,Positive
1,"Thank you for a wonderful evening in Washington, D.C. #Inauguration pic.twitter.com/a6xpFQTHj5",thank you for a wonderful evening in washington d c inauguration,0.7351,Positive
2,It all begins today! I will see you at 11:00 A.M. for the swearing-in. THE MOVEMENT CONTINUES - THE WORK BEGINS!,it all begins today i will see you at a m for the swearing in the movement continues the work begins,-0.2500,Negative
3,"Today we are not merely transferring power from one Administration to another, or from one party to another – but we are transferring...",today we are not merely transferring power from one administration to another or from one party to another but we are transferring,0.2144,Positive
4,"power from Washington, D.C. and giving it back to you, the American People. #InaugurationDay",power from washington d c and giving it back to you the american people inaugurationday,0.3400,Positive


## 9) Correct Market and Geopolitical Keyword Detection
This fixes the broken issue where `Market-Related Posts` was returning zero.

Design:
- Use word boundaries so `gas` does not match `vegas`.
- Keep phrases like `supply chain`, `interest rate`, and `crude oil`.
- Validate matches immediately after creating the feature.

In [10]:
market_keywords = [
    'oil', 'gas', 'energy', 'crude', 'crude oil', 'petroleum', 'opec',
    'saudi', 'iran', 'iraq', 'russia', 'china', 'war', 'missile',
    'sanction', 'sanctions', 'tariff', 'tariffs', 'trade', 'supply', 'supply chain',
    'shipping', 'inflation', 'prices', 'market', 'economy', 'economic',
    'jobs', 'dollar', 'fed', 'interest', 'interest rate', 'rates', 'stock', 'stocks',
    'exports', 'imports', 'manufacturing', 'pipeline', 'refinery', 'fuel'
]

geopolitical_keywords = [
    'iran', 'iraq', 'russia', 'china', 'saudi', 'opec', 'war', 'missile', 'military',
    'sanction', 'sanctions', 'tariff', 'tariffs', 'trade', 'border', 'conflict',
    'ukraine', 'nato', 'north korea', 'israel', 'gaza', 'red sea', 'houthi'
]

# Stronger subset focused directly on oil/energy.
oil_direct_keywords = [
    'oil', 'crude', 'crude oil', 'gas', 'energy', 'petroleum', 'opec', 'fuel',
    'pipeline', 'refinery', 'drilling', 'shale'
]

def compile_keyword_patterns(keywords):
    patterns = []
    for kw in sorted(set(keywords), key=len, reverse=True):
        escaped = re.escape(kw.lower()).replace('\\ ', r'\s+')
        pattern = re.compile(r'(?<![a-zA-Z])' + escaped + r'(?![a-zA-Z])', flags=re.IGNORECASE)
        patterns.append((kw, pattern))
    return patterns

market_patterns = compile_keyword_patterns(market_keywords)
geo_patterns = compile_keyword_patterns(geopolitical_keywords)
oil_direct_patterns = compile_keyword_patterns(oil_direct_keywords)

def keyword_count(text, patterns):
    text = str(text).lower()
    return sum(1 for _, pattern in patterns if pattern.search(text))

def matched_keywords(text, patterns):
    text = str(text).lower()
    return [kw for kw, pattern in patterns if pattern.search(text)]

df['market_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, market_patterns))
df['geo_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, geo_patterns))
df['oil_direct_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, oil_direct_patterns))

df['market_keywords_matched'] = df['clean_text'].apply(lambda x: ', '.join(matched_keywords(x, market_patterns)))
df['geo_keywords_matched'] = df['clean_text'].apply(lambda x: ', '.join(matched_keywords(x, geo_patterns)))

df['is_market_related'] = (df['market_keyword_count'] > 0).astype(int)
df['is_geo_related'] = (df['geo_keyword_count'] > 0).astype(int)
df['is_oil_direct_related'] = (df['oil_direct_keyword_count'] > 0).astype(int)

df['abs_polarity'] = df['polarity'].abs()
df['log_engagement'] = np.log1p(df['engagement'])

df['final_impact_sentiment'] = df['polarity'] * (1 + df['market_keyword_count']) * df['log_engagement']
df['market_importance'] = df['is_market_related'] * (1 + df['geo_keyword_count']) * df['log_engagement']
df['oil_direct_importance'] = df['is_oil_direct_related'] * (1 + df['geo_keyword_count']) * df['log_engagement']

summary_keyword_check = pd.DataFrame({
    'Metric': [
        'Market-related posts',
        'Geo-related posts',
        'Direct oil/energy posts',
        'Total market keyword mentions',
        'Total geo keyword mentions',
        'Total direct oil keyword mentions'
    ],
    'Value': [
        int(df['is_market_related'].sum()),
        int(df['is_geo_related'].sum()),
        int(df['is_oil_direct_related'].sum()),
        int(df['market_keyword_count'].sum()),
        int(df['geo_keyword_count'].sum()),
        int(df['oil_direct_keyword_count'].sum()),
    ]
})

display_dark_table(summary_keyword_check)

if df['is_market_related'].sum() == 0:
    raise ValueError('Market keyword detection failed: zero market-related posts found. Check regex/text cleaning.')

,Metric,Value
0,Market-related posts,3813
1,Geo-related posts,3711
2,Direct oil/energy posts,346
3,Total market keyword mentions,5355
4,Total geo keyword mentions,4603
5,Total direct oil keyword mentions,411


## 9.1) Added Improvement — Expanded Keyword Coverage
This cell extends the existing keyword system without removing the original logic. It adds more energy-market, macroeconomic, and geopolitical terms, then recalculates the keyword-based features used later in the notebook.


In [11]:
# =========================
# Added Improvement: Expanded Energy + Geopolitical Keyword System
# Place: directly after the original keyword detection cell
# =========================

energy_keywords_extra = [
    # Energy and oil market terms
    'brent', 'wti', 'crude', 'crude oil', 'oil prices', 'oil price',
    'gasoline', 'diesel', 'natural gas', 'lng', 'energy prices',
    'oil supply', 'oil demand', 'supply cut', 'production cut',
    'output cut', 'barrel', 'barrels', 'shale', 'drilling',
    'refinery', 'refineries', 'pipeline', 'pipelines',
    'strategic petroleum reserve', 'spr', 'opec+', 'opec plus'
]

geopolitical_keywords_extra = [
    # Geopolitical and supply-chain risk terms
    'middle east', 'persian gulf', 'strait of hormuz', 'red sea',
    'suez canal', 'yemen', 'houthi', 'houthis', 'gulf',
    'sanctions', 'embargo', 'tariff', 'tariffs',
    'trade war', 'conflict', 'military strike', 'missile attack',
    'ukraine', 'nato', 'russia', 'iran', 'saudi arabia',
    'china', 'israel', 'gaza'
]

macro_keywords_extra = [
    # Economic words that may affect oil indirectly
    'inflation', 'interest rate', 'rates', 'fed', 'federal reserve',
    'recession', 'growth', 'gdp', 'dollar', 'usd',
    'jobs report', 'unemployment', 'manufacturing',
    'trade deficit', 'exports', 'imports'
]

# Keep the original lists and only expand them.
market_keywords = sorted(set(market_keywords + energy_keywords_extra + geopolitical_keywords_extra + macro_keywords_extra))
geopolitical_keywords = sorted(set(geopolitical_keywords + geopolitical_keywords_extra))
oil_direct_keywords = sorted(set(oil_direct_keywords + energy_keywords_extra))

# Rebuild patterns using the original helper functions.
market_patterns = compile_keyword_patterns(market_keywords)
geo_patterns = compile_keyword_patterns(geopolitical_keywords)
oil_direct_patterns = compile_keyword_patterns(oil_direct_keywords)

# Recalculate keyword features using the expanded lists.
df['market_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, market_patterns))
df['geo_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, geo_patterns))
df['oil_direct_keyword_count'] = df['clean_text'].apply(lambda x: keyword_count(x, oil_direct_patterns))

df['market_keywords_matched'] = df['clean_text'].apply(lambda x: ', '.join(matched_keywords(x, market_patterns)))
df['geo_keywords_matched'] = df['clean_text'].apply(lambda x: ', '.join(matched_keywords(x, geo_patterns)))
df['oil_keywords_matched'] = df['clean_text'].apply(lambda x: ', '.join(matched_keywords(x, oil_direct_patterns)))

df['is_market_related'] = (df['market_keyword_count'] > 0).astype(int)
df['is_geo_related'] = (df['geo_keyword_count'] > 0).astype(int)
df['is_oil_direct_related'] = (df['oil_direct_keyword_count'] > 0).astype(int)

# Recalculate impact features so later daily aggregation uses the improved keyword detection.
df['final_impact_sentiment'] = df['polarity'] * (1 + df['market_keyword_count']) * df['log_engagement']
df['market_importance'] = df['is_market_related'] * (1 + df['geo_keyword_count']) * df['log_engagement']
df['oil_direct_importance'] = df['is_oil_direct_related'] * (1 + df['geo_keyword_count']) * df['log_engagement']

summary_keyword_check = pd.DataFrame({
    'Metric': [
        'Market-related posts',
        'Geo-related posts',
        'Direct oil/energy posts',
        'Total market keyword mentions',
        'Total geo keyword mentions',
        'Total direct oil keyword mentions'
    ],
    'Value': [
        int(df['is_market_related'].sum()),
        int(df['is_geo_related'].sum()),
        int(df['is_oil_direct_related'].sum()),
        int(df['market_keyword_count'].sum()),
        int(df['geo_keyword_count'].sum()),
        int(df['oil_direct_keyword_count'].sum()),
    ]
})

print('Expanded keyword system completed.')
print('Market keywords:', len(market_keywords))
print('Geopolitical keywords:', len(geopolitical_keywords))
print('Oil-direct keywords:', len(oil_direct_keywords))
display_dark_table(summary_keyword_check)

display_dark_table(
    df[['date', 'text', 'sentiment', 'market_keyword_count',
        'geo_keyword_count', 'oil_direct_keyword_count',
        'oil_keywords_matched']].head(10)
)


Expanded keyword system completed.
Market keywords: 92
Geopolitical keywords: 35
Oil-direct keywords: 34


,Metric,Value
0,Market-related posts,4340
1,Geo-related posts,3766
2,Direct oil/energy posts,367
3,Total market keyword mentions,6421
4,Total geo keyword mentions,4725
5,Total direct oil keyword mentions,484


,date,text,sentiment,market_keyword_count,geo_keyword_count,oil_direct_keyword_count,oil_keywords_matched
0,2017-01-20 00:40:51+00:00,"Thank you for joining us at the Lincoln Memorial tonight- a very special evening! Together, we are going to MAKE AMERICA GREAT AGAIN! pic.twitter.com/5d774OCx5o",Positive,0,0,0,
1,2017-01-20 04:24:33+00:00,"Thank you for a wonderful evening in Washington, D.C. #Inauguration pic.twitter.com/a6xpFQTHj5",Positive,0,0,0,
2,2017-01-20 12:31:53+00:00,It all begins today! I will see you at 11:00 A.M. for the swearing-in. THE MOVEMENT CONTINUES - THE WORK BEGINS!,Negative,0,0,0,
3,2017-01-20 17:51:25+00:00,"Today we are not merely transferring power from one Administration to another, or from one party to another – but we are transferring...",Positive,0,0,0,
4,2017-01-20 17:51:58+00:00,"power from Washington, D.C. and giving it back to you, the American People. #InaugurationDay",Positive,0,0,0,
5,2017-01-20 17:52:45+00:00,"What truly matters is not which party controls our government, but whether our government is controlled by the people.",Positive,0,0,0,
6,2017-01-20 17:53:17+00:00,"January 20th 2017, will be remembered as the day the people became the rulers of this nation again.",Neutral,0,0,0,
7,2017-01-20 17:54:00+00:00,"The forgotten men and women of our country will be forgotten no longer. From this moment on, it’s going to be #AmericaFirst🇺🇸",Negative,0,0,0,
8,2017-01-20 17:54:36+00:00,We will bring back our jobs. We will bring back our borders. We will bring back our wealth - and we will bring back our dreams!,Positive,1,0,0,
9,2017-01-20 17:55:44+00:00,We will follow two simple rules: BUY AMERICAN & HIRE AMERICAN! #InaugurationDay #MAGA🇺🇸,Neutral,0,0,0,


## 10) Inspect Market-Related Tweets
This cell verifies that the keyword feature is not silently broken.

In [12]:
market_examples = df[df['is_market_related'] == 1][
    ['date', 'platform', 'text', 'market_keywords_matched', 'geo_keywords_matched', 'polarity', 'sentiment']
].head(15)

display_dark_table(market_examples)

,date,platform,text,market_keywords_matched,geo_keywords_matched,polarity,sentiment
8,2017-01-20 17:54:36+00:00,Twitter,We will bring back our jobs. We will bring back our borders. We will bring back our wealth - and we will bring back our dreams!,jobs,,0.7096,Positive
22,2017-01-23 11:38:16+00:00,Twitter,Busy week planned with a heavy focus on jobs and national security. Top executives coming in at 9:00 A.M. to talk manufacturing in America.,"manufacturing, jobs",,0.4939,Positive
23,2017-01-24 11:11:47+00:00,Twitter,Will be meeting at 9:00 with top automobile executives concerning jobs in America. I want new plants to be built here for cars sold here!,jobs,,0.2732,Positive
26,2017-01-24 17:49:17+00:00,Twitter,Signing orders to move forward with the construction of the Keystone XL and Dakota Access pipelines in the Oval Office. pic.twitter.com/OErGmbBvYK – at The Oval Office,pipelines,,0.0000,Neutral
41,2017-01-26 13:51:46+00:00,Twitter,The U.S. has a 60 billion dollar trade deficit with Mexico. It has been a one-sided deal from the beginning of NAFTA with massive numbers...,"trade deficit, dollar, trade",trade,-0.4019,Negative
42,2017-01-26 13:55:03+00:00,Twitter,"of jobs and companies lost. If Mexico is unwilling to pay for the badly needed wall, then it would be better to cancel the upcoming meeting.",jobs,,-0.5994,Negative
48,2017-01-27 13:19:10+00:00,Twitter,"Mexico has taken advantage of the U.S. for long enough. Massive trade deficits & little help on the very weak border must change, NOW!",trade,"border, trade",0.0552,Positive
55,2017-01-28 13:08:42+00:00,Twitter,Thr coverage about me in the @nytimes and the @washingtonpost gas been so false and angry that the times actually apologized to its.....,gas,,-0.3102,Negative
61,2017-01-29 15:03:58+00:00,Twitter,Christians in the Middle-East have been executed in large numbers. We cannot allow this horror to continue!,middle east,middle east,-0.7220,Negative
63,2017-01-29 21:49:32+00:00,Twitter,"...Senators should focus their energies on ISIS, illegal immigration and border security instead of always looking to start World War III.",war,"border, war",-0.6369,Negative


## 11) EDA Overview
EDA can use all available historical data. Modeling evaluation will use only the chronological test period.

In [13]:
summary_cards = pd.DataFrame({
    'Metric': [
        'Total Posts', 'Start Date', 'End Date', 'Sentiment Engine',
        'Average Polarity', 'Market-Related Posts', 'Direct Oil/Energy Posts'
    ],
    'Value': [
        len(df),
        str(df['date'].min()),
        str(df['date'].max()),
        SENTIMENT_ENGINE,
        round(df['polarity'].mean(), 4),
        int(df['is_market_related'].sum()),
        int(df['is_oil_direct_related'].sum())
    ]
})

display_dark_table(summary_cards)

,Metric,Value
0,Total Posts,32094
1,Start Date,2017-01-20 00:40:51+00:00
2,End Date,2023-03-13 22:21:35+00:00
3,Sentiment Engine,VADER
4,Average Polarity,0.1240
5,Market-Related Posts,4340
6,Direct Oil/Energy Posts,367


In [14]:
if PLOTLY_OK:
    sentiment_counts = df['sentiment'].value_counts().reset_index()
    sentiment_counts.columns = ['sentiment', 'count']
    fig = px.pie(sentiment_counts, names='sentiment', values='count', title='Sentiment Distribution')
    style_fig(fig, height=450).show()
else:
    df['sentiment'].value_counts().plot(kind='pie', autopct='%1.1f%%', figsize=(6,6), title='Sentiment Distribution')
    plt.ylabel('')
    plt.show()

In [15]:
posts_by_year = df.groupby('year', as_index=False).agg(posts=('id', 'count'))

if PLOTLY_OK:
    fig = px.line(posts_by_year, x='year', y='posts', markers=True, title='Posts by Year')
    style_fig(fig).show()
else:
    posts_by_year.plot(x='year', y='posts', marker='o', title='Posts by Year')
    plt.show()

In [16]:
platform_counts = df['platform'].replace('', 'Unknown').value_counts().head(10).reset_index()
platform_counts.columns = ['platform', 'posts']

if PLOTLY_OK:
    fig = px.bar(platform_counts, x='platform', y='posts', title='Top Platforms')
    style_fig(fig).show()
else:
    platform_counts.plot(kind='bar', x='platform', y='posts', title='Top Platforms')
    plt.show()

In [17]:
market_by_year = df.groupby('year', as_index=False).agg(
    total_posts=('id', 'count'),
    market_posts=('is_market_related', 'sum'),
    geo_posts=('is_geo_related', 'sum'),
    oil_direct_posts=('is_oil_direct_related', 'sum')
)
market_by_year['market_share'] = market_by_year['market_posts'] / market_by_year['total_posts']

display_dark_table(market_by_year)

if PLOTLY_OK:
    melted = market_by_year.melt(
        id_vars='year',
        value_vars=['market_posts', 'geo_posts', 'oil_direct_posts'],
        var_name='Type',
        value_name='Posts'
    )
    fig = px.line(melted, x='year', y='Posts', color='Type', markers=True, title='Market/Geopolitical/Oil-Related Posts by Year')
    style_fig(fig).show()

,year,total_posts,market_posts,geo_posts,oil_direct_posts,market_share
0,2017,2472,409,312,20,0.1655
1,2018,3579,723,801,42,0.2020
2,2019,7841,1363,1229,92,0.1738
3,2020,12249,1273,939,90,0.1039
4,2021,156,5,3,0,0.0321
5,2022,4171,355,308,102,0.0851
6,2023,1626,212,174,21,0.1304


## 12) Top Hashtags
This is descriptive only and does not enter the model unless explicitly engineered.

In [18]:
all_hashtags = []
for text, hashtags in zip(df['text'], df['hashtags']):
    found = re.findall(r'#\w+', str(text).lower())
    if not found:
        found = re.findall(r'#\w+', str(hashtags).lower())
    all_hashtags.extend(found)

top_hashtags = pd.DataFrame(Counter(all_hashtags).most_common(20), columns=['hashtag', 'count'])
display_dark_table(top_hashtags)

if len(top_hashtags) > 0 and PLOTLY_OK:
    fig = px.bar(top_hashtags, x='count', y='hashtag', orientation='h', title='Top Hashtags')
    style_fig(fig).show()

,hashtag,count
0,#maga,472
1,#kag2020,77
2,#covid19,67
3,#1,49
4,#usmca,42
5,#2a,42
6,#americafirst,41
7,#usa,39
8,#coronavirus,38
9,#saveamericarally,36


## 13) Prepare Daily Tweet Features
All tweet signals are aggregated by day.

Important modeling assumption:
- Features from day **t** are used to predict oil movement on the **next trading day**.
- This is a daily forecasting setup, not intraday trading proof.

In [19]:
daily_tweets = df.groupby('date_day').agg(
    avg_sentiment=('sentiment_score', 'mean'),
    avg_polarity=('polarity', 'mean'),
    avg_subjectivity=('subjectivity', 'mean'),
    avg_positive_score=('positive_score', 'mean'),
    avg_neutral_score=('neutral_score', 'mean'),
    avg_negative_score=('negative_score', 'mean'),
    tweet_count=('text', 'count'),
    negative_tweets=('sentiment', lambda x: (x == 'Negative').sum()),
    positive_tweets=('sentiment', lambda x: (x == 'Positive').sum()),
    neutral_tweets=('sentiment', lambda x: (x == 'Neutral').sum()),
    total_likes=('favorite_count', 'sum'),
    total_reposts=('repost_count', 'sum'),
    total_engagement=('engagement', 'sum'),
    avg_word_count=('word_count', 'mean'),
    hashtag_count=('hashtag_count', 'sum'),
    mention_count=('mention_count', 'sum'),
    market_keyword_mentions=('market_keyword_count', 'sum'),
    geo_keyword_mentions=('geo_keyword_count', 'sum'),
    oil_direct_keyword_mentions=('oil_direct_keyword_count', 'sum'),
    market_related_posts=('is_market_related', 'sum'),
    geo_related_posts=('is_geo_related', 'sum'),
    oil_direct_related_posts=('is_oil_direct_related', 'sum'),
    avg_final_impact=('final_impact_sentiment', 'mean'),
    total_final_impact=('final_impact_sentiment', 'sum'),
    avg_market_importance=('market_importance', 'mean'),
    total_market_importance=('market_importance', 'sum'),
    avg_oil_direct_importance=('oil_direct_importance', 'mean'),
    total_oil_direct_importance=('oil_direct_importance', 'sum'),
).reset_index().rename(columns={'date_day': 'day'})

ratio_specs = [
    ('negative_tweets', 'tweet_count', 'negative_ratio'),
    ('positive_tweets', 'tweet_count', 'positive_ratio'),
    ('neutral_tweets', 'tweet_count', 'neutral_ratio'),
    ('market_related_posts', 'tweet_count', 'market_related_ratio'),
    ('geo_related_posts', 'tweet_count', 'geo_related_ratio'),
    ('oil_direct_related_posts', 'tweet_count', 'oil_direct_related_ratio'),
]

for numerator, denominator, new_col in ratio_specs:
    daily_tweets[new_col] = daily_tweets[numerator] / daily_tweets[denominator].replace(0, np.nan)

daily_tweets = daily_tweets.fillna(0)

print('Daily tweet features:', daily_tweets.shape)
display_dark_table(daily_tweets.head())

Daily tweet features: (1760, 35)


,day,avg_sentiment,avg_polarity,avg_subjectivity,avg_positive_score,avg_neutral_score,avg_negative_score,tweet_count,negative_tweets,positive_tweets,neutral_tweets,total_likes,total_reposts,total_engagement,avg_word_count,hashtag_count,mention_count,market_keyword_mentions,geo_keyword_mentions,oil_direct_keyword_mentions,market_related_posts,geo_related_posts,oil_direct_related_posts,avg_final_impact,total_final_impact,avg_market_importance,total_market_importance,avg_oil_direct_importance,total_oil_direct_importance,negative_ratio,positive_ratio,neutral_ratio,market_related_ratio,geo_related_ratio,oil_direct_related_ratio
0,2017-01-20 00:00:00+00:00,0.3077,0.1611,0.0000,0.1006,0.8685,0.0308,13,2,6,5,1638144.0000,350907.0000,1989051.0000,16.8462,5,0,1,0,0,1,0,0,2.5571,33.2420,0.9318,12.1133,0.0000,0.0000,0.1538,0.4615,0.3846,0.0769,0.0000,0.0000
1,2017-01-21 00:00:00+00:00,1.0000,0.8605,0.0000,0.3645,0.6355,0.0000,4,0,4,0,644697.0000,116191.0000,760888.0000,19.2500,0,1,0,0,0,0,0,0,10.2797,41.1187,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000
2,2017-01-22 00:00:00+00:00,0.6000,0.3696,0.0000,0.2550,0.6344,0.1104,5,1,4,0,962338.0000,187045.0000,1149383.0000,23.4000,0,1,0,0,0,0,0,0,4.4924,22.4622,0.0000,0.0000,0.0000,0.0000,0.2000,0.8000,0.0000,0.0000,0.0000,0.0000
3,2017-01-23 00:00:00+00:00,1.0000,0.4939,0.0000,0.1600,0.8400,0.0000,1,0,1,0,155135.0000,22161.0000,177296.0000,24.0000,0,0,2,0,0,1,0,0,17.9072,17.9072,12.0856,12.0856,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000
4,2017-01-24 00:00:00+00:00,0.7500,0.2438,0.0000,0.1227,0.8493,0.0280,4,0,3,1,431555.0000,71637.0000,503192.0000,21.0000,1,1,2,0,1,2,0,1,3.6558,14.6231,5.9416,23.7666,2.9541,11.8165,0.0000,0.7500,0.2500,0.5000,0.0000,0.2500


## 14) Prepare Daily Oil Data
Critical fix: the target is **next-day oil return**, not same-day return using same-day high/low/open/close as predictors.

In [20]:
def normalize_colnames(data):
    data = data.copy()
    data.columns = [str(c).strip() for c in data.columns]
    return data

def find_col(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    for c in columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c
    return None

oil_daily = normalize_colnames(oil_daily_raw)

date_col = find_col(oil_daily.columns, ['Date', 'Datetime', 'date', 'datetime'])
open_col = find_col(oil_daily.columns, ['Open', 'open'])
high_col = find_col(oil_daily.columns, ['High', 'high'])
low_col = find_col(oil_daily.columns, ['Low', 'low'])
close_col = find_col(oil_daily.columns, ['Close', 'Adj Close', 'close'])
volume_col = find_col(oil_daily.columns, ['Volume', 'volume'])

needed = {
    'date_col': date_col,
    'open_col': open_col,
    'high_col': high_col,
    'low_col': low_col,
    'close_col': close_col,
}
missing = [k for k, v in needed.items() if v is None]
if missing:
    raise ValueError(f'Could not detect required oil columns: {missing}. Available columns: {oil_daily.columns.tolist()}')

oil_daily = oil_daily.rename(columns={
    date_col: 'Date',
    open_col: 'oil_open',
    high_col: 'oil_high',
    low_col: 'oil_low',
    close_col: 'oil_close',
})

if volume_col is not None:
    oil_daily = oil_daily.rename(columns={volume_col: 'oil_volume'})
else:
    oil_daily['oil_volume'] = 0

oil_daily['Date'] = pd.to_datetime(oil_daily['Date'], errors='coerce', utc=True)
oil_daily = oil_daily.dropna(subset=['Date']).copy()
oil_daily['day'] = oil_daily['Date'].dt.floor('D')

for col in ['oil_open', 'oil_high', 'oil_low', 'oil_close', 'oil_volume']:
    oil_daily[col] = pd.to_numeric(oil_daily[col], errors='coerce')

oil_daily = oil_daily.dropna(subset=['oil_open', 'oil_high', 'oil_low', 'oil_close']).copy()
oil_daily = oil_daily.sort_values('day').drop_duplicates(subset=['day'], keep='last')

oil_daily['oil_return_pct_same_day'] = ((oil_daily['oil_close'] - oil_daily['oil_open']) / oil_daily['oil_open']) * 100
oil_daily['oil_delta_same_day'] = oil_daily['oil_close'] - oil_daily['oil_open']
oil_daily['oil_volatility_same_day'] = oil_daily['oil_high'] - oil_daily['oil_low']

for lag in [1, 2, 3, 5, 7, 10, 14]:
    oil_daily[f'oil_return_lag{lag}'] = oil_daily['oil_return_pct_same_day'].shift(lag)
    oil_daily[f'oil_close_lag{lag}'] = oil_daily['oil_close'].shift(lag)
    oil_daily[f'oil_volume_lag{lag}'] = oil_daily['oil_volume'].shift(lag)
    oil_daily[f'oil_volatility_lag{lag}'] = oil_daily['oil_volatility_same_day'].shift(lag)

for window in [3, 5, 7, 14]:
    oil_daily[f'oil_return_roll_mean_{window}'] = oil_daily['oil_return_pct_same_day'].shift(1).rolling(window, min_periods=1).mean()
    oil_daily[f'oil_return_roll_std_{window}'] = oil_daily['oil_return_pct_same_day'].shift(1).rolling(window, min_periods=2).std()
    oil_daily[f'oil_volatility_roll_mean_{window}'] = oil_daily['oil_volatility_same_day'].shift(1).rolling(window, min_periods=1).mean()

# Target: next available trading day movement.
oil_daily['target_next_day_return_pct'] = oil_daily['oil_return_pct_same_day'].shift(-1)
oil_daily['target_next_day_direction'] = (oil_daily['target_next_day_return_pct'] > 0).astype(int)
oil_daily['target_next_day'] = oil_daily['day'].shift(-1)

safe_oil_cols = ['day', 'target_next_day', 'target_next_day_return_pct', 'target_next_day_direction'] + [
    c for c in oil_daily.columns
    if c.startswith('oil_return_lag')
    or c.startswith('oil_close_lag')
    or c.startswith('oil_volume_lag')
    or c.startswith('oil_volatility_lag')
    or c.startswith('oil_return_roll')
    or c.startswith('oil_volatility_roll')
]

oil_model_daily = oil_daily[safe_oil_cols].copy()
print('Daily oil model data:', oil_model_daily.shape)
display_dark_table(oil_model_daily.head(10))

Daily oil model data: (2252, 44)


,day,target_next_day,target_next_day_return_pct,target_next_day_direction,oil_return_lag1,oil_close_lag1,oil_volume_lag1,oil_volatility_lag1,oil_return_lag2,oil_close_lag2,oil_volume_lag2,oil_volatility_lag2,oil_return_lag3,oil_close_lag3,oil_volume_lag3,oil_volatility_lag3,oil_return_lag5,oil_close_lag5,oil_volume_lag5,oil_volatility_lag5,oil_return_lag7,oil_close_lag7,oil_volume_lag7,oil_volatility_lag7,oil_return_lag10,oil_close_lag10,oil_volume_lag10,oil_volatility_lag10,oil_return_lag14,oil_close_lag14,oil_volume_lag14,oil_volatility_lag14,oil_return_roll_mean_3,oil_return_roll_std_3,oil_volatility_roll_mean_3,oil_return_roll_mean_5,oil_return_roll_std_5,oil_volatility_roll_mean_5,oil_return_roll_mean_7,oil_return_roll_std_7,oil_volatility_roll_mean_7,oil_return_roll_mean_14,oil_return_roll_std_14,oil_volatility_roll_mean_14
0,2017-01-20 00:00:00+00:00,2017-01-23 00:00:00+00:00,-1.0876,0,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—
1,2017-01-23 00:00:00+00:00,2017-01-24 00:00:00+00:00,0.6054,1,1.8853,52.4200,567231.0000,1.5100,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,1.8853,—,1.5100,1.8853,—,1.5100,1.8853,—,1.5100,1.8853,—,1.5100
2,2017-01-24 00:00:00+00:00,2017-01-25 00:00:00+00:00,-0.3777,0,-1.0876,52.7500,455333.0000,1.2600,1.8853,52.4200,567231.0000,1.5100,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,0.3989,2.1022,1.3850,0.3989,2.1022,1.3850,0.3989,2.1022,1.3850,0.3989,2.1022,1.3850
3,2017-01-25 00:00:00+00:00,2017-01-26 00:00:00+00:00,1.5483,1,0.6054,53.1800,520285.0000,0.8900,-1.0876,52.7500,455333.0000,1.2600,1.8853,52.4200,567231.0000,1.5100,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,0.4677,1.4912,1.2200,0.4677,1.4912,1.2200,0.4677,1.4912,1.2200,0.4677,1.4912,1.2200
4,2017-01-26 00:00:00+00:00,2017-01-27 00:00:00+00:00,-1.0607,0,-0.3777,52.7500,589709.0000,0.9100,0.6054,53.1800,520285.0000,0.8900,-1.0876,52.7500,455333.0000,1.2600,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,-0.2866,0.8501,1.0200,0.2564,1.2889,1.1425,0.2564,1.2889,1.1425,0.2564,1.2889,1.1425
5,2017-01-27 00:00:00+00:00,2017-01-30 00:00:00+00:00,-0.9784,0,1.5483,53.7800,578065.0000,1.2700,-0.3777,52.7500,589709.0000,0.9100,0.6054,53.1800,520285.0000,0.8900,1.8853,52.4200,567231.0000,1.5100,—,—,—,—,—,—,—,—,—,—,—,—,0.5920,0.9631,1.0233,0.5147,1.2569,1.1680,0.5147,1.2569,1.1680,0.5147,1.2569,1.1680
6,2017-01-30 00:00:00+00:00,2017-01-31 00:00:00+00:00,0.3992,1,-1.0607,53.1700,495556.0000,1.5000,1.5483,53.7800,578065.0000,1.2700,-0.3777,52.7500,589709.0000,0.9100,-1.0876,52.7500,455333.0000,1.2600,—,—,—,—,—,—,—,—,—,—,—,—,0.0367,1.3530,1.2267,-0.0744,1.1387,1.1660,0.2522,1.2952,1.2233,0.2522,1.2952,1.2233
7,2017-01-31 00:00:00+00:00,2017-02-01 00:00:00+00:00,2.1228,1,-0.9784,52.6300,486309.0000,1.0500,-1.0607,53.1700,495556.0000,1.5000,1.5483,53.7800,578065.0000,1.2700,0.6054,53.1800,520285.0000,0.8900,1.8853,52.4200,567231.0000,1.5100,—,—,—,—,—,—,—,—,-0.1636,1.4831,1.2733,-0.0526,1.1152,1.1240,0.0764,1.2705,1.1986,0.0764,1.2705,1.1986
8,2017-02-01 00:00:00+00:00,2017-02-02 00:00:00+00:00,-0.0560,0,0.3992,52.8100,586676.0000,1.3200,-0.9784,52.6300,486309.0000,1.0500,-1.0607,53.1700,495556.0000,1.5000,-0.3777,52.7500,589709.0000,0.9100,-1.0876,52.7500,455333.0000,1.2600,—,—,—,—,—,—,—,—,-0.5466,0.8202,1.2900,-0.0938,1.0883,1.2100,-0.1359,1.0167,1.1714,0.1167,1.1818,1.2138
9,2017-02-02 00:00:00+00:00,2017-02-03 00:00:00+00:00,0.2794,1,2.1228,53.8800,577243.0000,1.2700,0.3992,52.8100,586676.0000,1.3200,-0.9784,52.6300,486309.0000,1.0500,1.5483,53.7800,578065.0000,1.2700,0.6054,53.1800,520285.0000,0.8900,—,—,—,—,—,—,—,—,0.5146,1.5538,1.2133,0.4063,1.4422,1.2820,0.3227,1.2197,1.1729,0.3396,1.2920,1.2200


## 15) Merge Daily Tweets with Safe Future Oil Target
For day `t`, the model uses:
- tweet features from day `t`
- oil lag features from days before `t`

to predict oil movement on the next available trading day.

In [21]:
daily_merged = pd.merge(daily_tweets, oil_model_daily, on='day', how='inner')
daily_merged = daily_merged.sort_values('day').dropna(subset=['target_next_day_return_pct']).copy()

# Tweet lag features.
tweet_lag_base_cols = [
    'avg_sentiment', 'avg_polarity', 'tweet_count', 'negative_tweets', 'positive_tweets',
    'market_keyword_mentions', 'geo_keyword_mentions', 'oil_direct_keyword_mentions',
    'market_related_posts', 'geo_related_posts', 'oil_direct_related_posts',
    'total_engagement', 'total_final_impact', 'total_market_importance', 'total_oil_direct_importance'
]

tweet_lag_base_cols = [c for c in tweet_lag_base_cols if c in daily_merged.columns]

for lag in [1, 2, 3, 5, 7, 14]:
    for col in tweet_lag_base_cols:
        daily_merged[f'{col}_lag{lag}'] = daily_merged[col].shift(lag)

# Rolling features based only on previous rows.
rolling_base_cols = [
    'avg_sentiment', 'avg_polarity', 'tweet_count', 'market_keyword_mentions',
    'geo_keyword_mentions', 'oil_direct_keyword_mentions', 'total_engagement',
    'total_final_impact', 'total_market_importance', 'total_oil_direct_importance'
]
rolling_base_cols = [c for c in rolling_base_cols if c in daily_merged.columns]

for window in [3, 7, 14]:
    for col in rolling_base_cols:
        daily_merged[f'{col}_roll_mean_{window}'] = daily_merged[col].shift(1).rolling(window=window, min_periods=1).mean()
        daily_merged[f'{col}_roll_sum_{window}'] = daily_merged[col].shift(1).rolling(window=window, min_periods=1).sum()

# Drop rows created by lags/rolling oil std.
daily_merged = daily_merged.replace([np.inf, -np.inf], np.nan).dropna().copy()

print('Merged daily modeling rows:', daily_merged.shape)
print('Date range:', daily_merged['day'].min(), 'to', daily_merged['day'].max())
display_dark_table(daily_merged.head())

Merged daily modeling rows: (1200, 228)
Date range: 2017-02-09 00:00:00+00:00 to 2023-03-13 00:00:00+00:00


,day,avg_sentiment,avg_polarity,avg_subjectivity,avg_positive_score,avg_neutral_score,avg_negative_score,tweet_count,negative_tweets,positive_tweets,neutral_tweets,total_likes,total_reposts,total_engagement,avg_word_count,hashtag_count,mention_count,market_keyword_mentions,geo_keyword_mentions,oil_direct_keyword_mentions,market_related_posts,geo_related_posts,oil_direct_related_posts,avg_final_impact,total_final_impact,avg_market_importance,total_market_importance,avg_oil_direct_importance,total_oil_direct_importance,negative_ratio,positive_ratio,neutral_ratio,market_related_ratio,geo_related_ratio,oil_direct_related_ratio,target_next_day,target_next_day_return_pct,target_next_day_direction,oil_return_lag1,oil_close_lag1,oil_volume_lag1,oil_volatility_lag1,oil_return_lag2,oil_close_lag2,oil_volume_lag2,oil_volatility_lag2,oil_return_lag3,oil_close_lag3,oil_volume_lag3,oil_volatility_lag3,oil_return_lag5,oil_close_lag5,oil_volume_lag5,oil_volatility_lag5,oil_return_lag7,oil_close_lag7,oil_volume_lag7,oil_volatility_lag7,oil_return_lag10,oil_close_lag10,oil_volume_lag10,oil_volatility_lag10,oil_return_lag14,oil_close_lag14,oil_volume_lag14,oil_volatility_lag14,oil_return_roll_mean_3,oil_return_roll_std_3,oil_volatility_roll_mean_3,oil_return_roll_mean_5,oil_return_roll_std_5,oil_volatility_roll_mean_5,oil_return_roll_mean_7,oil_return_roll_std_7,oil_volatility_roll_mean_7,oil_return_roll_mean_14,oil_return_roll_std_14,oil_volatility_roll_mean_14,avg_sentiment_lag1,avg_polarity_lag1,tweet_count_lag1,negative_tweets_lag1,positive_tweets_lag1,market_keyword_mentions_lag1,geo_keyword_mentions_lag1,oil_direct_keyword_mentions_lag1,market_related_posts_lag1,geo_related_posts_lag1,oil_direct_related_posts_lag1,total_engagement_lag1,total_final_impact_lag1,total_market_importance_lag1,total_oil_direct_importance_lag1,avg_sentiment_lag2,avg_polarity_lag2,tweet_count_lag2,negative_tweets_lag2,positive_tweets_lag2,market_keyword_mentions_lag2,geo_keyword_mentions_lag2,oil_direct_keyword_mentions_lag2,market_related_posts_lag2,geo_related_posts_lag2,oil_direct_related_posts_lag2,total_engagement_lag2,total_final_impact_lag2,total_market_importance_lag2,total_oil_direct_importance_lag2,avg_sentiment_lag3,avg_polarity_lag3,tweet_count_lag3,negative_tweets_lag3,positive_tweets_lag3,market_keyword_mentions_lag3,geo_keyword_mentions_lag3,oil_direct_keyword_mentions_lag3,market_related_posts_lag3,geo_related_posts_lag3,oil_direct_related_posts_lag3,total_engagement_lag3,total_final_impact_lag3,total_market_importance_lag3,total_oil_direct_importance_lag3,avg_sentiment_lag5,avg_polarity_lag5,tweet_count_lag5,negative_tweets_lag5,positive_tweets_lag5,market_keyword_mentions_lag5,geo_keyword_mentions_lag5,oil_direct_keyword_mentions_lag5,market_related_posts_lag5,geo_related_posts_lag5,oil_direct_related_posts_lag5,total_engagement_lag5,total_final_impact_lag5,total_market_importance_lag5,total_oil_direct_importance_lag5,avg_sentiment_lag7,avg_polarity_lag7,tweet_count_lag7,negative_tweets_lag7,positive_tweets_lag7,market_keyword_mentions_lag7,geo_keyword_mentions_lag7,oil_direct_keyword_mentions_lag7,market_related_posts_lag7,geo_related_posts_lag7,oil_direct_related_posts_lag7,total_engagement_lag7,total_final_impact_lag7,total_market_importance_lag7,total_oil_direct_importance_lag7,avg_sentiment_lag14,avg_polarity_lag14,tweet_count_lag14,negative_tweets_lag14,positive_tweets_lag14,market_keyword_mentions_lag14,geo_keyword_mentions_lag14,oil_direct_keyword_mentions_lag14,market_related_posts_lag14,geo_related_posts_lag14,oil_direct_related_posts_lag14,total_engagement_lag14,total_final_impact_lag14,total_market_importance_lag14,total_oil_direct_importance_lag14,avg_sentiment_roll_mean_3,avg_sentiment_roll_sum_3,avg_polarity_roll_mean_3,avg_polarity_roll_sum_3,tweet_count_roll_mean_3,tweet_count_roll_sum_3,market_keyword_mentions_roll_mean_3,market_keyword_mentions_roll_sum_3,geo_keyword_mentions_roll_mean_3,geo_keyword_mentions_roll_sum_3,oil_dire

## 15.1) Added Improvement — Sentiment Shock Analysis
This cell checks whether extreme positive or negative tweet-sentiment days are linked with different next-day oil returns and volatility-related signals.


In [22]:
# =========================
# Added Improvement: Sentiment Shock Features + Oil Volatility Link
# Place: directly after the daily_merged creation cell
# =========================

shock_df = daily_merged.copy()

# Define sentiment shock thresholds using quantiles.
neg_threshold = shock_df['avg_sentiment'].quantile(0.10)
pos_threshold = shock_df['avg_sentiment'].quantile(0.90)

shock_df['negative_sentiment_shock'] = shock_df['avg_sentiment'] <= neg_threshold
shock_df['positive_sentiment_shock'] = shock_df['avg_sentiment'] >= pos_threshold

# Detect volatility columns already created in the notebook.
volatility_cols = [c for c in shock_df.columns if 'oil_volatility' in c.lower()]

print('Negative sentiment threshold:', round(neg_threshold, 4))
print('Positive sentiment threshold:', round(pos_threshold, 4))
print('Available volatility columns:')
print(volatility_cols)

agg_dict = {
    'days': ('day', 'count'),
    'avg_next_day_return': ('target_next_day_return_pct', 'mean'),
    'median_next_day_return': ('target_next_day_return_pct', 'median'),
    'avg_tweet_count': ('tweet_count', 'mean'),
    'avg_negative_tweets': ('negative_tweets', 'mean'),
    'avg_positive_tweets': ('positive_tweets', 'mean'),
    'avg_market_mentions': ('market_keyword_mentions', 'mean'),
    'avg_oil_mentions': ('oil_direct_keyword_mentions', 'mean')
}

# Add the first available volatility column to the summary if it exists.
if volatility_cols:
    agg_dict['avg_oil_volatility_proxy'] = (volatility_cols[0], 'mean')

shock_summary = shock_df.groupby(
    ['negative_sentiment_shock', 'positive_sentiment_shock']
).agg(**agg_dict).reset_index()

display_dark_table(shock_summary)


Negative sentiment threshold: -0.1667
Positive sentiment threshold: 0.6692
Available volatility columns:
['oil_volatility_lag1', 'oil_volatility_lag2', 'oil_volatility_lag3', 'oil_volatility_lag5', 'oil_volatility_lag7', 'oil_volatility_lag10', 'oil_volatility_lag14', 'oil_volatility_roll_mean_3', 'oil_volatility_roll_mean_5', 'oil_volatility_roll_mean_7', 'oil_volatility_roll_mean_14']


,negative_sentiment_shock,positive_sentiment_shock,days,avg_next_day_return,median_next_day_return,avg_tweet_count,avg_negative_tweets,avg_positive_tweets,avg_market_mentions,avg_oil_mentions,avg_oil_volatility_proxy
0,False,False,957,-0.4664,0.1199,20.7806,5.7670,9.9195,4.1630,0.3083,2.1924
1,False,True,120,-0.1365,0.0811,6.6583,0.2250,5.7417,1.7917,0.0667,1.7811
2,True,False,123,-0.2826,0.0555,12.5447,6.7236,2.8455,2.7317,0.1057,2.3047


## 16) Leakage Audit
This blocks the most dangerous mistake: using current-day/future oil columns as predictors.

In [23]:
leakage_forbidden_features = [
    'oil_open', 'oil_high', 'oil_low', 'oil_close', 'oil_volume',
    'oil_return_pct_same_day', 'oil_delta_same_day', 'oil_volatility_same_day',
    'target_next_day_return_pct', 'target_next_day_direction', 'target_next_day'
]

print('Forbidden raw/current/future oil columns that must NOT be in X:')
print(leakage_forbidden_features)

Forbidden raw/current/future oil columns that must NOT be in X:
['oil_open', 'oil_high', 'oil_low', 'oil_close', 'oil_volume', 'oil_return_pct_same_day', 'oil_delta_same_day', 'oil_volatility_same_day', 'target_next_day_return_pct', 'target_next_day_direction', 'target_next_day']


## 17) Chronological Train/Test Split
Time series data must be split by time. Older dates train the model; newer dates test it.

In [24]:
def chronological_train_test_split(data, date_col='day', test_size=0.2):
    data = data.sort_values(date_col).copy()
    split_idx = int(len(data) * (1 - test_size))
    train = data.iloc[:split_idx].copy()
    test = data.iloc[split_idx:].copy()
    if len(train) == 0 or len(test) == 0:
        raise ValueError('Train or test split is empty. Check dataset size or test_size.')
    if train[date_col].max() >= test[date_col].min():
        raise ValueError('Chronological split failed: train period overlaps test period.')
    return train, test

train_df, test_df = chronological_train_test_split(daily_merged, date_col='day', test_size=TEST_SIZE)

print('Train rows:', train_df.shape[0])
print('Test rows:', test_df.shape[0])
print('Train period:', train_df['day'].min(), 'to', train_df['day'].max())
print('Test period:', test_df['day'].min(), 'to', test_df['day'].max())
print('Chronological split check:', train_df['day'].max() < test_df['day'].min())

Train rows: 960
Test rows: 240
Train period: 2017-02-09 00:00:00+00:00 to 2020-12-07 00:00:00+00:00
Test period: 2020-12-08 00:00:00+00:00 to 2023-03-13 00:00:00+00:00
Chronological split check: True


## 18) Distribution Shift Check
This matters because Trump moved from Twitter to Truth Social. If the platform distribution changes, model performance can drop even if the code is correct.

In [25]:
platform_daily = df.groupby(['date_day', 'platform']).size().reset_index(name='posts')
platform_pivot = platform_daily.pivot_table(index='date_day', columns='platform', values='posts', fill_value=0).reset_index()
platform_pivot = platform_pivot.rename(columns={'date_day': 'day'})

platform_cols = [c for c in platform_pivot.columns if c != 'day']
platform_pivot['dominant_platform'] = platform_pivot[platform_cols].idxmax(axis=1) if platform_cols else 'Unknown'

train_platform = platform_pivot[platform_pivot['day'].isin(train_df['day'])]['dominant_platform'].value_counts(normalize=True).rename('Train Share')
test_platform = platform_pivot[platform_pivot['day'].isin(test_df['day'])]['dominant_platform'].value_counts(normalize=True).rename('Test Share')
platform_shift = pd.concat([train_platform, test_platform], axis=1).fillna(0)

display_dark_table(platform_shift)

,Train Share,Test Share
dominant_platform,,
Twitter,1.0000,0.0875
Truth Social,0.0000,0.9125


## 19) Define Feature Sets
We build three feature groups:

1. **Dummy**: no real features, just a benchmark.
2. **Oil-only baseline**: previous oil behavior only.
3. **Tweet/Sentiment model**: oil lags + tweet/sentiment/market features.

The question is not “Can we get any accuracy?” The question is: **do tweet features beat the oil-only baseline?**

In [26]:
baseline_features = [
    c for c in daily_merged.columns
    if c.startswith('oil_return_lag')
    or c.startswith('oil_close_lag')
    or c.startswith('oil_volume_lag')
    or c.startswith('oil_volatility_lag')
    or c.startswith('oil_return_roll')
    or c.startswith('oil_volatility_roll')
]

same_day_tweet_features = [
    'avg_sentiment', 'avg_polarity', 'avg_subjectivity', 'avg_positive_score', 'avg_neutral_score', 'avg_negative_score',
    'tweet_count', 'negative_tweets', 'positive_tweets', 'neutral_tweets',
    'negative_ratio', 'positive_ratio', 'neutral_ratio',
    'total_likes', 'total_reposts', 'total_engagement',
    'avg_word_count', 'hashtag_count', 'mention_count',
    'market_keyword_mentions', 'geo_keyword_mentions', 'oil_direct_keyword_mentions',
    'market_related_posts', 'geo_related_posts', 'oil_direct_related_posts',
    'market_related_ratio', 'geo_related_ratio', 'oil_direct_related_ratio',
    'avg_final_impact', 'total_final_impact',
    'avg_market_importance', 'total_market_importance',
    'avg_oil_direct_importance', 'total_oil_direct_importance'
]
same_day_tweet_features = [c for c in same_day_tweet_features if c in daily_merged.columns]

lagged_tweet_features = [
    c for c in daily_merged.columns
    if any(c.endswith(f'_lag{lag}') for lag in [1, 2, 3, 5, 7, 14])
    or any(c.endswith(f'_roll_mean_{window}') for window in [3, 7, 14])
    or any(c.endswith(f'_roll_sum_{window}') for window in [3, 7, 14])
]

sentiment_features = baseline_features + same_day_tweet_features + lagged_tweet_features

# Remove forbidden columns in case they accidentally entered.
baseline_features = [c for c in baseline_features if c not in leakage_forbidden_features]
sentiment_features = [c for c in sentiment_features if c not in leakage_forbidden_features]

bad_base = sorted(set(baseline_features).intersection(leakage_forbidden_features))
bad_sent = sorted(set(sentiment_features).intersection(leakage_forbidden_features))
assert not bad_base, f'Leakage in baseline features: {bad_base}'
assert not bad_sent, f'Leakage in sentiment features: {bad_sent}'

print('Baseline feature count:', len(baseline_features))
print('Sentiment feature count:', len(sentiment_features))
print('\nBaseline features:')
print(baseline_features)
print('\nFirst 40 sentiment features:')
print(sentiment_features[:40])

Baseline feature count: 40
Sentiment feature count: 254

Baseline features:
['oil_return_lag1', 'oil_close_lag1', 'oil_volume_lag1', 'oil_volatility_lag1', 'oil_return_lag2', 'oil_close_lag2', 'oil_volume_lag2', 'oil_volatility_lag2', 'oil_return_lag3', 'oil_close_lag3', 'oil_volume_lag3', 'oil_volatility_lag3', 'oil_return_lag5', 'oil_close_lag5', 'oil_volume_lag5', 'oil_volatility_lag5', 'oil_return_lag7', 'oil_close_lag7', 'oil_volume_lag7', 'oil_volatility_lag7', 'oil_return_lag10', 'oil_close_lag10', 'oil_volume_lag10', 'oil_volatility_lag10', 'oil_return_lag14', 'oil_close_lag14', 'oil_volume_lag14', 'oil_volatility_lag14', 'oil_return_roll_mean_3', 'oil_return_roll_std_3', 'oil_volatility_roll_mean_3', 'oil_return_roll_mean_5', 'oil_return_roll_std_5', 'oil_volatility_roll_mean_5', 'oil_return_roll_mean_7', 'oil_return_roll_std_7', 'oil_volatility_roll_mean_7', 'oil_return_roll_mean_14', 'oil_return_roll_std_14', 'oil_volatility_roll_mean_14']

First 40 sentiment features:
['oil

## 20) Prepare X and y Safely
The scaler is fitted only on training data, then applied to test data.

In [27]:
from sklearn.preprocessing import StandardScaler

Xb_train = train_df[baseline_features].copy()
Xb_test = test_df[baseline_features].copy()
Xs_train = train_df[sentiment_features].copy()
Xs_test = test_df[sentiment_features].copy()

y_reg_train = train_df['target_next_day_return_pct'].copy()
y_reg_test = test_df['target_next_day_return_pct'].copy()

y_clf_train = train_df['target_next_day_direction'].copy()
y_clf_test = test_df['target_next_day_direction'].copy()

for data_part in [Xb_train, Xb_test, Xs_train, Xs_test]:
    data_part.replace([np.inf, -np.inf], np.nan, inplace=True)
    data_part.fillna(0, inplace=True)

scaler_base = StandardScaler()
scaler_sent = StandardScaler()

Xb_train_scaled = scaler_base.fit_transform(Xb_train)
Xb_test_scaled = scaler_base.transform(Xb_test)

Xs_train_scaled = scaler_sent.fit_transform(Xs_train)
Xs_test_scaled = scaler_sent.transform(Xs_test)

print('Scaling done using training data only.')
print('Train class balance:')
print(y_clf_train.value_counts(normalize=True).rename('share'))
print('\nTest class balance:')
print(y_clf_test.value_counts(normalize=True).rename('share'))

Scaling done using training data only.
Train class balance:
target_next_day_direction
1    0.523958
0    0.476042
Name: share, dtype: float64

Test class balance:
target_next_day_direction
1    0.508333
0    0.491667
Name: share, dtype: float64


## 21) Regression Models — Predict Next-Day Oil Return %
This answers: can the features estimate the size of next-day oil return?

Expected reality: oil returns are noisy, so regression may be weak. Negative R² is possible and should not be hidden.

In [28]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

regression_models = {
    'Dummy Mean': ('baseline', DummyRegressor(strategy='mean')),
    'Oil Baseline Ridge': ('baseline', Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    'Oil Baseline RF': ('baseline', RandomForestRegressor(n_estimators=250, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)),
    'Sentiment Ridge': ('sentiment', Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    'Sentiment Huber': ('sentiment', HuberRegressor(max_iter=500)),
    'Sentiment RF': ('sentiment', RandomForestRegressor(n_estimators=250, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)),
    'Sentiment GB': ('sentiment', GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=RANDOM_STATE)),
}

reg_results = []
reg_predictions = {}

for name, (feature_type, model) in regression_models.items():
    if feature_type == 'baseline':
        model.fit(Xb_train_scaled, y_reg_train)
        pred = model.predict(Xb_test_scaled)
    else:
        model.fit(Xs_train_scaled, y_reg_train)
        pred = model.predict(Xs_test_scaled)

    reg_predictions[name] = pred
    rmse = mean_squared_error(y_reg_test, pred) ** 0.5
    reg_results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_reg_test, pred),
        'RMSE': rmse,
        'R2': r2_score(y_reg_test, pred)
    })

reg_results_df = pd.DataFrame(reg_results).sort_values('R2', ascending=False)
display_dark_table(reg_results_df)

,Model,MAE,RMSE,R2
0,Dummy Mean,2.0340,2.5364,-0.0205
4,Sentiment Huber,2.1652,2.7600,-0.2083
6,Sentiment GB,2.2574,2.8516,-0.2898
2,Oil Baseline RF,3.2215,4.2607,-1.8794
5,Sentiment RF,3.3899,4.4734,-2.1741
1,Oil Baseline Ridge,6.1722,7.8063,-8.6658
3,Sentiment Ridge,6.4409,8.2840,-9.8851


## 22) Regression Interpretation Guardrail
This cell prevents overclaiming.

In [29]:
best_reg = reg_results_df.iloc[0]
print('Best regression model:', best_reg['Model'])
print('Best R2:', round(best_reg['R2'], 4))
print('Best MAE:', round(best_reg['MAE'], 4))

if best_reg['R2'] < 0:
    print('\nInterpretation: Regression performance is weak. The model does not explain next-day oil return better than a simple average benchmark.')
elif best_reg['R2'] < 0.1:
    print('\nInterpretation: Regression signal exists but is weak. Do not overclaim predictive power.')
else:
    print('\nInterpretation: Regression has some useful signal, but still needs business caution and external variables.')

if PLOTLY_OK:
    fig = px.bar(reg_results_df, x='Model', y='R2', text='R2', title='Regression Model Comparison — R²')
    fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    style_fig(fig, height=520).show()
else:
    reg_results_df.plot(kind='bar', x='Model', y='R2', title='Regression Model Comparison — R²')
    plt.xticks(rotation=45)
    plt.show()

Best regression model: Dummy Mean
Best R2: -0.0205
Best MAE: 2.034

Interpretation: Regression performance is weak. The model does not explain next-day oil return better than a simple average benchmark.


## 23) Classification Models — Predict Next-Day Up/Down Direction
This answers: can the features predict whether oil goes up or down tomorrow?

Balanced Accuracy is emphasized because raw accuracy can be misleading when classes are imbalanced.

In [30]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, roc_auc_score
)

classification_models = {
    'Dummy Majority': ('baseline', DummyClassifier(strategy='most_frequent')),
    'Oil Baseline Logistic': ('baseline', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
    'Oil Baseline RF': ('baseline', RandomForestClassifier(n_estimators=250, max_depth=5, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    'Sentiment Logistic': ('sentiment', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
    'Sentiment RF': ('sentiment', RandomForestClassifier(n_estimators=250, max_depth=5, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    'Sentiment GB': ('sentiment', GradientBoostingClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=RANDOM_STATE)),
}

clf_results = []
clf_predictions = {}
clf_probabilities = {}

for name, (feature_type, model) in classification_models.items():
    if feature_type == 'baseline':
        model.fit(Xb_train_scaled, y_clf_train)
        pred = model.predict(Xb_test_scaled)
        if hasattr(model, 'predict_proba'):
            prob = model.predict_proba(Xb_test_scaled)[:, 1]
        else:
            prob = None
    else:
        model.fit(Xs_train_scaled, y_clf_train)
        pred = model.predict(Xs_test_scaled)
        if hasattr(model, 'predict_proba'):
            prob = model.predict_proba(Xs_test_scaled)[:, 1]
        else:
            prob = None

    clf_predictions[name] = pred
    clf_probabilities[name] = prob

    try:
        auc = roc_auc_score(y_clf_test, prob) if prob is not None else np.nan
    except Exception:
        auc = np.nan

    clf_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_clf_test, pred),
        'Balanced Accuracy': balanced_accuracy_score(y_clf_test, pred),
        'Precision': precision_score(y_clf_test, pred, zero_division=0),
        'Recall': recall_score(y_clf_test, pred, zero_division=0),
        'F1': f1_score(y_clf_test, pred, zero_division=0),
        'ROC AUC': auc
    })

clf_results_df = pd.DataFrame(clf_results).sort_values('Balanced Accuracy', ascending=False)
display_dark_table(clf_results_df)

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC AUC
3,Sentiment Logistic,0.5125,0.5076,0.5131,0.8033,0.6262,0.4803
0,Dummy Majority,0.5083,0.5000,0.5083,1.0000,0.6740,0.5000
5,Sentiment GB,0.4917,0.4910,0.5000,0.5328,0.5159,0.4920
4,Sentiment RF,0.4958,0.4888,0.5023,0.9098,0.6472,0.4654
2,Oil Baseline RF,0.4750,0.4757,0.4818,0.4344,0.4569,0.4778
1,Oil Baseline Logistic,0.4417,0.4430,0.4400,0.3607,0.3964,0.4353


## 24) Classification Report for Best Model
This shows exactly where the model fails: false positives, false negatives, and class imbalance.

In [31]:
best_clf_name = clf_results_df.iloc[0]['Model']
best_clf_pred = clf_predictions[best_clf_name]

print('Best classification model:', best_clf_name)
print('\nClassification report:')
print(classification_report(y_clf_test, best_clf_pred, target_names=['Down/Flat', 'Up'], zero_division=0))

cm = confusion_matrix(y_clf_test, best_clf_pred)
cm_df = pd.DataFrame(cm, index=['Actual Down/Flat', 'Actual Up'], columns=['Pred Down/Flat', 'Pred Up'])
display_dark_table(cm_df)

if PLOTLY_OK:
    fig = px.imshow(cm_df, text_auto=True, title=f'Confusion Matrix — {best_clf_name}')
    style_fig(fig, height=500).show()

Best classification model: Sentiment Logistic

Classification report:
              precision    recall  f1-score   support

   Down/Flat       0.51      0.21      0.30       118
          Up       0.51      0.80      0.63       122

    accuracy                           0.51       240
   macro avg       0.51      0.51      0.46       240
weighted avg       0.51      0.51      0.47       240



,Pred Down/Flat,Pred Up
Actual Down/Flat,25,93
Actual Up,24,98


## 24.1) Added Improvement — Stronger Quantitative Model Evaluation
This cell expands the classification evaluation beyond accuracy, which is important because financial direction prediction can be noisy and class balance can be imperfect.


In [32]:
# =========================
# Added Improvement: Stronger Quantitative Model Evaluation
# Place: directly after the best classification model report
# =========================

strong_eval_rows = []

for model_name, preds in clf_predictions.items():
    probs = clf_probabilities.get(model_name)

    row = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_clf_test, preds),
        'Balanced Accuracy': balanced_accuracy_score(y_clf_test, preds),
        'Precision': precision_score(y_clf_test, preds, zero_division=0),
        'Recall': recall_score(y_clf_test, preds, zero_division=0),
        'F1 Score': f1_score(y_clf_test, preds, zero_division=0)
    }

    if probs is not None:
        try:
            row['ROC AUC'] = roc_auc_score(y_clf_test, probs)
        except Exception:
            row['ROC AUC'] = np.nan
    else:
        row['ROC AUC'] = np.nan

    strong_eval_rows.append(row)

strong_eval_df = pd.DataFrame(strong_eval_rows).sort_values(
    by=['Balanced Accuracy', 'F1 Score'],
    ascending=False
)

display_dark_table(strong_eval_df)

if PLOTLY_OK:
    metric_viz = strong_eval_df.melt(
        id_vars='Model',
        value_vars=['Accuracy', 'Balanced Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC'],
        var_name='Metric',
        value_name='Score'
    )

    fig = px.bar(
        metric_viz,
        x='Model',
        y='Score',
        color='Metric',
        barmode='group',
        title='Full Classification Model Evaluation'
    )

    fig.update_layout(xaxis_tickangle=-35, height=600)
    style_fig(fig, height=600).show()


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1 Score,ROC AUC
3,Sentiment Logistic,0.5125,0.5076,0.5131,0.8033,0.6262,0.4803
0,Dummy Majority,0.5083,0.5000,0.5083,1.0000,0.6740,0.5000
5,Sentiment GB,0.4917,0.4910,0.5000,0.5328,0.5159,0.4920
4,Sentiment RF,0.4958,0.4888,0.5023,0.9098,0.6472,0.4654
2,Oil Baseline RF,0.4750,0.4757,0.4818,0.4344,0.4569,0.4778
1,Oil Baseline Logistic,0.4417,0.4430,0.4400,0.3607,0.3964,0.4353


## 25) Honest Baseline Comparison
This directly checks whether tweet/sentiment features add value beyond the oil-only baseline.

In [33]:
baseline_only = clf_results_df[clf_results_df['Model'].str.startswith('Oil Baseline')].sort_values('Balanced Accuracy', ascending=False)
sentiment_only = clf_results_df[clf_results_df['Model'].str.startswith('Sentiment')].sort_values('Balanced Accuracy', ascending=False)
dummy_only = clf_results_df[clf_results_df['Model'].str.startswith('Dummy')].head(1)

comparison_summary = pd.concat([dummy_only, baseline_only.head(1), sentiment_only.head(1)], ignore_index=True)
display_dark_table(comparison_summary)

if not baseline_only.empty and not sentiment_only.empty:
    base_score = baseline_only.iloc[0]['Balanced Accuracy']
    sent_score = sentiment_only.iloc[0]['Balanced Accuracy']
    diff = sent_score - base_score
    print(f'Sentiment improvement over oil-only baseline Balanced Accuracy: {diff:.4f}')

    if diff > 0.03:
        print('Interpretation: Tweet/sentiment features add a meaningful predictive improvement over the oil-only baseline.')
    elif diff > 0.01:
        print('Interpretation: Tweet/sentiment features add a small but visible improvement. Still avoid strong claims.')
    elif diff > 0:
        print('Interpretation: Tweet/sentiment features improve the baseline slightly, but the gain is very small.')
    else:
        print('Interpretation: Tweet/sentiment features do not beat the oil-only baseline. Do not overclaim.')

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC AUC
0,Dummy Majority,0.5083,0.5000,0.5083,1.0000,0.6740,0.5000
1,Oil Baseline RF,0.4750,0.4757,0.4818,0.4344,0.4569,0.4778
2,Sentiment Logistic,0.5125,0.5076,0.5131,0.8033,0.6262,0.4803


Sentiment improvement over oil-only baseline Balanced Accuracy: 0.0319
Interpretation: Tweet/sentiment features add a meaningful predictive improvement over the oil-only baseline.


## 26) Model Comparison Dashboard

In [34]:
if PLOTLY_OK:
    metric_cols = ['Accuracy', 'Balanced Accuracy', 'F1', 'ROC AUC']
    melted = clf_results_df.melt(id_vars='Model', value_vars=metric_cols, var_name='Metric', value_name='Score')
    fig = px.bar(melted, x='Model', y='Score', color='Metric', barmode='group', title='Classification Model Comparison')
    style_fig(fig, height=600).show()
else:
    clf_results_df.plot(kind='bar', x='Model', y=['Accuracy', 'Balanced Accuracy', 'F1'], title='Classification Model Comparison')
    plt.xticks(rotation=45)
    plt.show()

## 27) Correlation Analysis — Correctly Framed
Correlations are descriptive. They do not prove causality.

In [35]:
corr_cols = [
    'target_next_day_return_pct',
    'avg_sentiment', 'avg_polarity', 'tweet_count', 'negative_tweets', 'positive_tweets',
    'market_keyword_mentions', 'geo_keyword_mentions', 'oil_direct_keyword_mentions',
    'market_related_posts', 'geo_related_posts', 'oil_direct_related_posts',
    'total_engagement', 'avg_final_impact', 'total_final_impact',
    'total_market_importance', 'total_oil_direct_importance'
]
corr_cols = [c for c in corr_cols if c in daily_merged.columns]

corr_df = daily_merged[corr_cols].corr(numeric_only=True)[['target_next_day_return_pct']].sort_values('target_next_day_return_pct', ascending=False)
display_dark_table(corr_df)

if PLOTLY_OK:
    plot_corr = corr_df.drop(index='target_next_day_return_pct', errors='ignore').reset_index()
    plot_corr.columns = ['Feature', 'Correlation']
    fig = px.bar(plot_corr, x='Correlation', y='Feature', orientation='h', title='Correlation with Next-Day Oil Return')
    style_fig(fig, height=650).show()

,target_next_day_return_pct
target_next_day_return_pct,1.0000
geo_related_posts,0.0268
geo_keyword_mentions,0.0264
avg_final_impact,0.0218
total_final_impact,0.0214
market_keyword_mentions,0.0165
oil_direct_keyword_mentions,0.0156
total_market_importance,0.0149
oil_direct_related_posts,0.0146
avg_polarity,0.0125


## 27.1) Added Improvement — Event Study Around Sentiment Shocks
This cell adds a stronger analytical layer by checking average next-day oil returns around extreme positive and negative sentiment days. This supports cautious association analysis, not direct causality claims.


In [36]:
# =========================
# Added Improvement: Event Study Around Strong Sentiment Shock Days
# Place: directly after the correlation analysis cell
# =========================

event_df = daily_merged.copy().sort_values('day').reset_index(drop=True)

# Define strong sentiment events using the 10th and 90th percentiles.
negative_event_threshold = event_df['avg_sentiment'].quantile(0.10)
positive_event_threshold = event_df['avg_sentiment'].quantile(0.90)

negative_event_days = event_df.loc[
    event_df['avg_sentiment'] <= negative_event_threshold, 'day'
].tolist()

positive_event_days = event_df.loc[
    event_df['avg_sentiment'] >= positive_event_threshold, 'day'
].tolist()

def build_event_window(data, event_days, label, window=3):
    rows = []

    for event_day in event_days:
        event_index = data.index[data['day'] == event_day]

        if len(event_index) == 0:
            continue

        event_index = event_index[0]

        for offset in range(-window, window + 1):
            idx = event_index + offset

            if idx < 0 or idx >= len(data):
                continue

            rows.append({
                'event_day': event_day,
                'relative_day': offset,
                'event_type': label,
                'day': data.loc[idx, 'day'],
                'target_next_day_return_pct': data.loc[idx, 'target_next_day_return_pct'],
                'avg_sentiment': data.loc[idx, 'avg_sentiment'],
                'negative_tweets': data.loc[idx, 'negative_tweets'],
                'positive_tweets': data.loc[idx, 'positive_tweets'],
                'market_keyword_mentions': data.loc[idx, 'market_keyword_mentions'],
                'oil_direct_keyword_mentions': data.loc[idx, 'oil_direct_keyword_mentions']
            })

    return pd.DataFrame(rows)

negative_window = build_event_window(event_df, negative_event_days, 'Negative Sentiment Shock', window=3)
positive_window = build_event_window(event_df, positive_event_days, 'Positive Sentiment Shock', window=3)

event_study_df = pd.concat([negative_window, positive_window], ignore_index=True)

event_summary = event_study_df.groupby(['event_type', 'relative_day']).agg(
    avg_next_day_return=('target_next_day_return_pct', 'mean'),
    median_next_day_return=('target_next_day_return_pct', 'median'),
    avg_sentiment=('avg_sentiment', 'mean'),
    avg_market_mentions=('market_keyword_mentions', 'mean'),
    observations=('target_next_day_return_pct', 'count')
).reset_index()

display_dark_table(event_summary)

if PLOTLY_OK:
    fig = px.line(
        event_summary,
        x='relative_day',
        y='avg_next_day_return',
        color='event_type',
        markers=True,
        title='Event Study: Average Next-Day Oil Return Around Sentiment Shocks'
    )

    fig.add_vline(x=0, line_dash='dash')
    fig.update_layout(
        xaxis_title='Days Around Sentiment Shock',
        yaxis_title='Average Next-Day Oil Return %',
        height=550
    )

    style_fig(fig, height=550).show()


,event_type,relative_day,avg_next_day_return,median_next_day_return,avg_sentiment,avg_market_mentions,observations
0,Negative Sentiment Shock,-3,-0.2778,-0.3045,0.2002,3.5285,123
1,Negative Sentiment Shock,-2,0.2907,0.3865,0.1648,3.3740,123
2,Negative Sentiment Shock,-1,-0.0070,0.2969,0.0929,3.2439,123
3,Negative Sentiment Shock,0,-0.2826,0.0555,-0.3635,2.7317,123
4,Negative Sentiment Shock,1,-2.7487,0.0811,0.1246,3.1951,123
5,Negative Sentiment Shock,2,-1.4271,0.0891,0.1340,3.4228,123
6,Negative Sentiment Shock,3,-0.0836,0.0705,0.1512,3.4065,123
7,Positive Sentiment Shock,-3,-1.2427,0.1528,0.3421,2.5126,119
8,Positive Sentiment Shock,-2,0.2535,0.2051,0.3895,2.6555,119
9,Positive Sentiment Shock,-1,0.0161,0.0960,0.3831,2.6134,119


## 28) Fixed Visualization Scaling
Sentiment and oil return are not on the same scale. This chart uses z-scores for visual comparison only.

In [37]:
def zscore(s):
    s = pd.Series(s)
    std = s.std()
    if std == 0 or pd.isna(std):
        return s * 0
    return (s - s.mean()) / std

viz_df = daily_merged[['day', 'avg_sentiment', 'target_next_day_return_pct', 'market_keyword_mentions', 'oil_direct_keyword_mentions']].copy()
viz_df['avg_sentiment_z'] = zscore(viz_df['avg_sentiment'])
viz_df['next_day_return_z'] = zscore(viz_df['target_next_day_return_pct'])
viz_df['market_mentions_z'] = zscore(viz_df['market_keyword_mentions'])
viz_df['oil_direct_mentions_z'] = zscore(viz_df['oil_direct_keyword_mentions'])

if PLOTLY_OK:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=viz_df['day'], y=viz_df['avg_sentiment_z'], mode='lines', name='Avg Sentiment z-score'))
    fig.add_trace(go.Scatter(x=viz_df['day'], y=viz_df['next_day_return_z'], mode='lines', name='Next-Day Oil Return z-score'))
    fig.add_trace(go.Scatter(x=viz_df['day'], y=viz_df['market_mentions_z'], mode='lines', name='Market Keyword Mentions z-score', opacity=0.55))
    style_fig(fig, title='Normalized Sentiment / Market Mentions vs Next-Day Oil Return', height=560).show()
else:
    plt.figure(figsize=(14,5))
    plt.plot(viz_df['day'], viz_df['avg_sentiment_z'], label='Avg Sentiment z-score')
    plt.plot(viz_df['day'], viz_df['next_day_return_z'], label='Next-Day Return z-score')
    plt.plot(viz_df['day'], viz_df['market_mentions_z'], label='Market Mentions z-score', alpha=0.6)
    plt.legend()
    plt.title('Normalized Sentiment / Market Mentions vs Next-Day Oil Return')
    plt.show()

## 28.1) Added Improvement — Clearer Sentiment/Oil Visualizations
These charts make the relationship between negative/positive tweets, market/oil mentions, oil returns, and volatility easier to interpret than a single normalized line chart.


In [38]:
# =========================
# Added Improvement: Positive/Negative Tweets vs Next-Day Oil Return
# Place: directly after the fixed visualization scaling cell
# =========================

viz_sentiment_oil = daily_merged[
    ['day', 'negative_tweets', 'positive_tweets',
     'negative_ratio', 'positive_ratio',
     'target_next_day_return_pct',
     'market_keyword_mentions', 'oil_direct_keyword_mentions']
].copy()

if PLOTLY_OK:
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=viz_sentiment_oil['day'],
        y=viz_sentiment_oil['negative_tweets'],
        name='Negative Tweets',
        opacity=0.65
    ))

    fig.add_trace(go.Bar(
        x=viz_sentiment_oil['day'],
        y=viz_sentiment_oil['positive_tweets'],
        name='Positive Tweets',
        opacity=0.65
    ))

    fig.add_trace(go.Scatter(
        x=viz_sentiment_oil['day'],
        y=viz_sentiment_oil['target_next_day_return_pct'],
        name='Next-Day Oil Return %',
        mode='lines',
        yaxis='y2'
    ))

    fig.update_layout(
        title='Negative/Positive Tweets vs Next-Day Oil Return',
        xaxis_title='Date',
        yaxis=dict(title='Tweet Count'),
        yaxis2=dict(
            title='Next-Day Oil Return %',
            overlaying='y',
            side='right'
        ),
        barmode='group',
        height=600
    )

    style_fig(fig, height=600).show()

else:
    plt.figure(figsize=(14, 6))
    plt.bar(viz_sentiment_oil['day'], viz_sentiment_oil['negative_tweets'], label='Negative Tweets', alpha=0.6)
    plt.bar(viz_sentiment_oil['day'], viz_sentiment_oil['positive_tweets'], label='Positive Tweets', alpha=0.6)
    plt.plot(viz_sentiment_oil['day'], viz_sentiment_oil['target_next_day_return_pct'], label='Next-Day Oil Return %')
    plt.legend()
    plt.title('Negative/Positive Tweets vs Next-Day Oil Return')
    plt.show()


In [39]:
# =========================
# Added Improvement: Market/Oil Mentions vs Oil Volatility Proxy
# Place: directly after the fixed visualization scaling cell
# =========================

vol_cols = [c for c in daily_merged.columns if 'oil_volatility' in c.lower()]
volatility_col = vol_cols[0] if len(vol_cols) > 0 else None

if volatility_col:
    vol_viz = daily_merged[
        ['day', 'market_keyword_mentions', 'oil_direct_keyword_mentions', volatility_col]
    ].copy()

    if PLOTLY_OK:
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=vol_viz['day'],
            y=vol_viz['market_keyword_mentions'],
            mode='lines',
            name='Market Keyword Mentions'
        ))

        fig.add_trace(go.Scatter(
            x=vol_viz['day'],
            y=vol_viz['oil_direct_keyword_mentions'],
            mode='lines',
            name='Oil-Direct Keyword Mentions'
        ))

        fig.add_trace(go.Scatter(
            x=vol_viz['day'],
            y=vol_viz[volatility_col],
            mode='lines',
            name=f'{volatility_col}',
            yaxis='y2'
        ))

        fig.update_layout(
            title='Market/Oil Keyword Mentions vs Oil Volatility',
            xaxis_title='Date',
            yaxis=dict(title='Keyword Mentions'),
            yaxis2=dict(
                title='Oil Volatility Proxy',
                overlaying='y',
                side='right'
            ),
            height=600
        )

        style_fig(fig, height=600).show()

    else:
        plt.figure(figsize=(14, 6))
        plt.plot(vol_viz['day'], vol_viz['market_keyword_mentions'], label='Market Keyword Mentions')
        plt.plot(vol_viz['day'], vol_viz['oil_direct_keyword_mentions'], label='Oil-Direct Mentions')
        plt.plot(vol_viz['day'], vol_viz[volatility_col], label=volatility_col)
        plt.legend()
        plt.title('Market/Oil Keyword Mentions vs Oil Volatility')
        plt.show()

else:
    print('No oil volatility column found in daily_merged.')


### Added Explanation for Report / Presentation

To strengthen the project, additional keyword categories were added to capture more oil-specific, energy-market, macroeconomic, and geopolitical terms. This improves the ability of the model to distinguish general political posts from tweets that are more likely to be relevant to oil market movements.

The visualization section was also improved by directly comparing negative and positive tweet counts with next-day oil returns, and by comparing market/oil keyword mentions with oil volatility proxies. These plots make the relationship between tweet sentiment and oil behavior easier to interpret.

In addition, an event-study approach was added. Instead of relying only on simple correlation, the analysis identifies extreme positive and negative sentiment days and examines oil returns around those events. This provides stronger analytical value, although it should still be interpreted carefully as association rather than proven causality.

Finally, the model evaluation was expanded beyond accuracy by including balanced accuracy, precision, recall, F1 score, and ROC AUC. This gives a more honest view of model performance, especially because financial direction prediction can be imbalanced and noisy.


## 29) Feature Importance for Tree-Based Sentiment Model
Feature importance is not causality. It only shows what the model used for prediction.

In [40]:
# Choose the best tree-based sentiment classifier if available.
tree_model_name = None
for candidate in ['Sentiment RF', 'Sentiment GB']:
    if candidate in classification_models:
        tree_model_name = candidate
        break

if tree_model_name is not None:
    feature_type, tree_model = classification_models[tree_model_name]
    tree_model.fit(Xs_train_scaled, y_clf_train)

    if hasattr(tree_model, 'feature_importances_'):
        importance_df = pd.DataFrame({
            'Feature': sentiment_features,
            'Importance': tree_model.feature_importances_
        }).sort_values('Importance', ascending=False).head(25)

        display_dark_table(importance_df)

        if PLOTLY_OK:
            fig = px.bar(importance_df, x='Importance', y='Feature', orientation='h', title=f'Top Feature Importances — {tree_model_name}')
            style_fig(fig, height=700).show()

,Feature,Importance
131,total_final_impact_lag2,0.0157
215,avg_sentiment_roll_sum_7,0.0102
84,oil_volume_lag3,0.0098
234,avg_sentiment_roll_mean_14,0.0096
53,total_likes,0.0092
120,avg_polarity_lag2,0.0091
214,avg_sentiment_roll_mean_7,0.0090
69,total_final_impact,0.0090
249,total_final_impact_roll_sum_14,0.0090
55,total_engagement,0.0088


## 30) Prediction Timeline for Best Classifier
This helps visually inspect when the model gets direction right or wrong.

In [41]:
test_predictions_df = test_df[['day', 'target_next_day', 'target_next_day_return_pct', 'target_next_day_direction']].copy()
test_predictions_df['predicted_direction'] = best_clf_pred
test_predictions_df['correct'] = (test_predictions_df['predicted_direction'] == test_predictions_df['target_next_day_direction']).astype(int)

display_dark_table(test_predictions_df.head(20))

if PLOTLY_OK:
    plot_df = test_predictions_df.copy()
    plot_df['Actual'] = plot_df['target_next_day_direction'].map({0: 'Down/Flat', 1: 'Up'})
    plot_df['Predicted'] = plot_df['predicted_direction'].map({0: 'Down/Flat', 1: 'Up'})
    fig = px.scatter(
        plot_df,
        x='day',
        y='target_next_day_return_pct',
        color='correct',
        hover_data=['Actual', 'Predicted'],
        title='Best Classifier: Correct vs Incorrect Predictions Over Time'
    )
    style_fig(fig, height=560).show()

,day,target_next_day,target_next_day_return_pct,target_next_day_direction,predicted_direction,correct
974,2020-12-08 00:00:00+00:00,2020-12-09 00:00:00+00:00,-0.1754,0,1,0
975,2020-12-09 00:00:00+00:00,2020-12-10 00:00:00+00:00,2.3856,1,1,1
976,2020-12-10 00:00:00+00:00,2020-12-11 00:00:00+00:00,-0.8516,0,1,0
977,2020-12-11 00:00:00+00:00,2020-12-14 00:00:00+00:00,0.5564,1,1,1
978,2020-12-14 00:00:00+00:00,2020-12-15 00:00:00+00:00,1.3407,1,0,0
979,2020-12-15 00:00:00+00:00,2020-12-16 00:00:00+00:00,0.4622,1,1,1
980,2020-12-16 00:00:00+00:00,2020-12-17 00:00:00+00:00,1.0658,1,1,1
981,2020-12-17 00:00:00+00:00,2020-12-18 00:00:00+00:00,1.3834,1,1,1
982,2020-12-18 00:00:00+00:00,2020-12-21 00:00:00+00:00,-1.6481,0,1,0
983,2020-12-21 00:00:00+00:00,2020-12-22 00:00:00+00:00,-1.8986,0,1,0


## 31) Error Analysis by Market Activity
Does the classifier perform better on days with more market-related tweets?

In [42]:
error_analysis_df = test_predictions_df.merge(
    daily_merged[['day', 'tweet_count', 'market_keyword_mentions', 'geo_keyword_mentions', 'oil_direct_keyword_mentions', 'market_related_posts']],
    on='day',
    how='left'
)

error_analysis_df['market_activity_bucket'] = pd.qcut(
    error_analysis_df['market_keyword_mentions'].rank(method='first'),
    q=3,
    labels=['Low Market Mentions', 'Medium Market Mentions', 'High Market Mentions']
)

bucket_perf = error_analysis_df.groupby('market_activity_bucket', observed=True).agg(
    rows=('correct', 'count'),
    accuracy=('correct', 'mean'),
    avg_market_mentions=('market_keyword_mentions', 'mean'),
    avg_oil_direct_mentions=('oil_direct_keyword_mentions', 'mean')
).reset_index()

display_dark_table(bucket_perf)

if PLOTLY_OK:
    fig = px.bar(bucket_perf, x='market_activity_bucket', y='accuracy', text='accuracy', title='Prediction Accuracy by Market-Mention Activity')
    fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    style_fig(fig, height=500).show()

,market_activity_bucket,rows,accuracy,avg_market_mentions,avg_oil_direct_mentions
0,Low Market Mentions,80,0.4000,0.0125,0.0000
1,Medium Market Mentions,80,0.6125,1.6375,0.1500
2,High Market Mentions,80,0.5250,6.1375,0.8875


## 32) Optional Hourly Analysis
This section is optional and descriptive. It should not be mixed with the daily model unless you carefully align timestamps.

Important fix: do not automatically choose Brent/WTI by row count if your daily target is WTI/CL=F. This notebook defaults to **WTI** when available.

In [43]:
if oil_hourly_raw is None:
    print('No hourly oil file found. Skipping hourly analysis.')
else:
    oil_hourly = normalize_colnames(oil_hourly_raw)
    datetime_col = find_col(oil_hourly.columns, ['datetime', 'Date', 'date', 'time'])
    close_h_col = find_col(oil_hourly.columns, ['close', 'Close'])
    oil_type_col = find_col(oil_hourly.columns, ['oil_type', 'type', 'name', 'ticker'])

    if datetime_col is None or close_h_col is None:
        print('Hourly file found, but required datetime/close columns were not detected. Skipping hourly analysis.')
    else:
        oil_hourly = oil_hourly.rename(columns={datetime_col: 'datetime', close_h_col: 'hourly_close'})
        oil_hourly['datetime'] = pd.to_datetime(oil_hourly['datetime'], errors='coerce', utc=True)
        oil_hourly['hourly_close'] = pd.to_numeric(oil_hourly['hourly_close'], errors='coerce')
        oil_hourly = oil_hourly.dropna(subset=['datetime', 'hourly_close']).copy()

        if oil_type_col is not None:
            oil_hourly = oil_hourly.rename(columns={oil_type_col: 'oil_type'})
            available_types = oil_hourly['oil_type'].astype(str).str.upper().unique().tolist()
            print('Available hourly oil types:', available_types)

            if any('WTI' in t for t in available_types):
                chosen_type = [t for t in available_types if 'WTI' in t][0]
                oil_hourly_selected = oil_hourly[oil_hourly['oil_type'].astype(str).str.upper() == chosen_type].copy()
                print('Chosen hourly oil type:', chosen_type)
            else:
                chosen_type = oil_hourly['oil_type'].value_counts().idxmax()
                oil_hourly_selected = oil_hourly[oil_hourly['oil_type'] == chosen_type].copy()
                print('WTI not found. Chosen dominant oil type:', chosen_type)
        else:
            oil_hourly_selected = oil_hourly.copy()
            chosen_type = 'Unknown'

        oil_hourly_selected = oil_hourly_selected.sort_values('datetime')
        oil_hourly_selected['hourly_return_pct'] = oil_hourly_selected['hourly_close'].pct_change() * 100
        oil_hourly_selected['hour'] = oil_hourly_selected['datetime'].dt.floor('H')

        print('Hourly selected shape:', oil_hourly_selected.shape)
        display_dark_table(oil_hourly_selected.head())

Available hourly oil types: ['BRENT', 'WTI']
Chosen hourly oil type: WTI
Hourly selected shape: (39837, 9)


,datetime,open,high,low,hourly_close,volume,oil_type,hourly_return_pct,hour
1,2017-01-20 00:00:00+00:00,52.3000,52.3100,52.2400,52.2900,0,WTI,—,2017-01-20 00:00:00+00:00
3,2017-01-20 01:00:00+00:00,52.2800,52.2800,52.1400,52.1600,0,WTI,-0.2486,2017-01-20 01:00:00+00:00
5,2017-01-20 02:00:00+00:00,52.1500,52.3500,52.1200,52.3500,0,WTI,0.3643,2017-01-20 02:00:00+00:00
7,2017-01-20 03:00:00+00:00,52.3600,52.4900,52.3300,52.4100,0,WTI,0.1146,2017-01-20 03:00:00+00:00
9,2017-01-20 04:00:00+00:00,52.4200,52.7900,52.3000,52.7400,0,WTI,0.6297,2017-01-20 04:00:00+00:00


## 33) Final Project Summary
This summary is generated from the actual model outputs.

In [44]:
final_summary = []

final_summary.append(f'Dataset period: {daily_merged["day"].min()} to {daily_merged["day"].max()}')
final_summary.append(f'Sentiment engine used: {SENTIMENT_ENGINE}')
final_summary.append(f'Market-related posts detected: {int(df["is_market_related"].sum())}')
final_summary.append(f'Direct oil/energy posts detected: {int(df["is_oil_direct_related"].sum())}')
final_summary.append(f'Best regression model: {best_reg["Model"]} | R2={best_reg["R2"]:.4f} | MAE={best_reg["MAE"]:.4f}')
final_summary.append(f'Best classification model: {best_clf_name}')
final_summary.append(f'Best classification balanced accuracy: {clf_results_df.iloc[0]["Balanced Accuracy"]:.4f}')

if not baseline_only.empty and not sentiment_only.empty:
    final_summary.append(f'Sentiment improvement over best oil-only baseline balanced accuracy: {diff:.4f}')

print('========== FINAL PROJECT SUMMARY ==========')
for item in final_summary:
    print('-', item)

print('\nHonest interpretation:')
if best_reg['R2'] < 0:
    print('- Regression is weak; it does not reliably estimate next-day oil return size.')
else:
    print('- Regression has some signal, but should still be interpreted cautiously.')

if not baseline_only.empty and not sentiment_only.empty:
    if diff > 0.03:
        print('- Tweet/sentiment features add a meaningful directional improvement over oil-only baseline.')
    elif diff > 0:
        print('- Tweet/sentiment features add only a small directional improvement over oil-only baseline.')
    else:
        print('- Tweet/sentiment features do not beat the oil-only baseline.')

print('- This project is strongest as an EDA + weak-signal predictive experiment, not as proof that tweets control oil prices.')

========== FINAL PROJECT SUMMARY ==========
- Dataset period: 2017-02-09 00:00:00+00:00 to 2023-03-13 00:00:00+00:00
- Sentiment engine used: VADER
- Market-related posts detected: 4340
- Direct oil/energy posts detected: 367
- Best regression model: Dummy Mean | R2=-0.0205 | MAE=2.0340
- Best classification model: Sentiment Logistic
- Best classification balanced accuracy: 0.5076
- Sentiment improvement over best oil-only baseline balanced accuracy: 0.0319

Honest interpretation:
- Regression is weak; it does not reliably estimate next-day oil return size.
- Tweet/sentiment features add a meaningful directional improvement over oil-only baseline.
- This project is strongest as an EDA + weak-signal predictive experiment, not as proof that tweets control oil prices.


## 34) Save Key Outputs
This exports the main tables for reporting.

In [45]:
OUTPUT_DIR = Path('project_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

summary_cards.to_csv(OUTPUT_DIR / 'summary_cards.csv', index=False)
summary_keyword_check.to_csv(OUTPUT_DIR / 'keyword_detection_check.csv', index=False)
reg_results_df.to_csv(OUTPUT_DIR / 'regression_results.csv', index=False)
clf_results_df.to_csv(OUTPUT_DIR / 'classification_results.csv', index=False)
comparison_summary.to_csv(OUTPUT_DIR / 'baseline_comparison.csv', index=False)
corr_df.to_csv(OUTPUT_DIR / 'correlation_with_next_day_return.csv')
test_predictions_df.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

print('Saved outputs to:', OUTPUT_DIR.resolve())

Saved outputs to: /content/project_outputs


# End of Notebook

Recommended presentation wording:

> The model shows that tweet-based sentiment and market-signal features may contain a weak short-term directional signal for oil movement, but the improvement over oil-only baselines is small. Therefore, the results should be interpreted as evidence of limited association, not causal control or strong predictive power.